# Sincronizzazione e Analisi Video Termici — Metodo 3 + Metodo 5 + Maschera Binaria Anomalie

Questa è una copia modificata di `conformal.ipynb`. Il notebook originale non viene toccato.

La modifica principale riguarda la vecchia fase di **Allineamento Temporale (Troncamento)**: invece di tagliare ogni mini-video ai primi `L` frame, ogni passata viene trattata come una funzione continua nel tempo/fase e ricampionata su una griglia comune. In questo modo tutti i mini-video hanno lo stesso numero di frame, ma viene usata l'intera passata, compresa la coda che prima veniva eliminata.

Sono mantenute due letture distinte:

- **tempo reale completo**: usa `dati/mini_video_roi/`, con lunghezze variabili, utile per studiare `T(t)` senza deformare la durata reale;
- **fase normalizzata ricampionata**: usa `dati/mini_video_resampled_phase/`, con lunghezza fissa, utile per modello conformal e confronto passata-su-passata.


In [16]:
import os
import json
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# File e percorsi
FRAME_DIR = os.path.join("dati", "frame_thermo")
MAP_FILE = "map.json"

# Le configurazioni ROI non vengono piu' salvate nella root del progetto.
# Nuova posizione ordinata: dati/config/roi/
ROI_CONFIG_DIR = os.path.join("dati", "config", "roi")
os.makedirs(ROI_CONFIG_DIR, exist_ok=True)

ROI_FILE_LEGACY = "roi_config.json"
ROI_FILE_601_LEGACY = "roi_config_601.json"

ROI_FILE = os.path.join(ROI_CONFIG_DIR, "roi_config.json")                 # ROI standard, usata per tutti i provini tranne 601
ROI_FILE_601 = os.path.join(ROI_CONFIG_DIR, "roi_config_601.json")         # ROI dedicata al provino 601, video Rec-G3_S60


def migrate_roi_config(old_path, new_path, label):
    """Sposta una vecchia configurazione ROI dalla root alla nuova cartella dati/config/roi/."""
    if os.path.exists(new_path):
        if os.path.exists(old_path):
            print(f"[INFO] {label}: uso la configurazione in {new_path}.")
            print(f"       Il vecchio file in root esiste ancora: {old_path}. Puoi eliminarlo dopo aver verificato che tutto funzioni.")
        return

    if os.path.exists(old_path):
        shutil.move(old_path, new_path)
        print(f"[OK] {label}: spostata da {old_path} a {new_path}")


migrate_roi_config(ROI_FILE_LEGACY, ROI_FILE, "ROI standard")
migrate_roi_config(ROI_FILE_601_LEGACY, ROI_FILE_601, "ROI 601")

# Il video termico del provino 601 puo' comparire con nomi diversi a seconda dell'estrazione frame.
# In output lo trattiamo sempre come provino "601", cosi' resta compatibile con i dati meccanici.
SPECIAL_PROVINO_601 = "601"
SPECIAL_601_ALIASES = ["601", "Rec-G3_S60", "Rec-G3_S60.seq", "G3_S60"]

# Se map.json non contiene 601/Rec-G3_S60, puoi impostare qui manualmente il frame di start.
# Lascia None per usare 0 come fallback.
MANUAL_SYNC_STARTS = {
    "601": None,
}


def canonical_provino_id(video_id):
    """Converte eventuali alias del video termico nel codice provino usato nei risultati."""
    v = str(video_id).replace(".seq", "")
    if v in [a.replace(".seq", "") for a in SPECIAL_601_ALIASES]:
        return SPECIAL_PROVINO_601
    return v


def is_provino_601(video_id):
    return canonical_provino_id(video_id) == SPECIAL_PROVINO_601


# Caricamento della mappa per la sincronizzazione
with open(MAP_FILE, 'r') as f:
    map_data = json.load(f)

sync_map = {str(item['id']).replace(".seq", ""): int(item['start']) for item in map_data}


def sync_candidates(video_id):
    """Restituisce i possibili identificativi da cercare in map.json."""
    v = str(video_id).replace(".seq", "")
    candidates = [v, canonical_provino_id(v)]
    if is_provino_601(v):
        candidates.extend([a.replace(".seq", "") for a in SPECIAL_601_ALIASES])
    # mantieni ordine e rimuovi duplicati
    return list(dict.fromkeys(candidates))


def get_sync_start(video_id):
    """Start frame robusto: prova source id, id canonico, alias 601, poi fallback manuale/0."""
    for candidate in sync_candidates(video_id):
        if candidate in sync_map:
            return sync_map[candidate]
    canonical = canonical_provino_id(video_id)
    manual = MANUAL_SYNC_STARTS.get(canonical, None)
    return int(manual) if manual is not None else 0


# Lista dei video disponibili gia' estratti in dati/frame_thermo.
# Nota: se Rec-G3_S60 non compare qui, prima va eseguita l'estrazione .seq -> PNG/tracking.
if os.path.exists(FRAME_DIR):
    available_videos = sorted([d for d in os.listdir(FRAME_DIR) if os.path.isdir(os.path.join(FRAME_DIR, d))])
else:
    available_videos = []

print(f"Video disponibili in {FRAME_DIR}: {available_videos}")

source_601 = [v for v in available_videos if is_provino_601(v)]
if source_601:
    print(f"[OK] Sorgente termica riconosciuta per provino 601: {source_601}")
else:
    print("[INFO] Provino 601 non ancora trovato in dati/frame_thermo.")
    print("       Dopo l'estrazione del file Rec-G3_S60.seq, la cartella puo' chiamarsi 601, Rec-G3_S60 o G3_S60.")



Video disponibili in dati\frame_thermo: ['101', '102', '301', '302', '303', '304', '601', '602', '901', '902', '992']
[OK] Sorgente termica riconosciuta per provino 601: ['601']


In [17]:
def get_thermal_metadata(video_id):
    meta_path = os.path.join(FRAME_DIR, str(video_id), "thermal_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path, "r") as f:
            return json.load(f)
    return None


def get_frames(video_id):
    video_path = os.path.join(FRAME_DIR, str(video_id))
    if not os.path.exists(video_path):
        return []
    return sorted([f for f in os.listdir(video_path) if f.endswith(".png")])


def load_roi_file(path):
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARN] Impossibile leggere {path}: {e}")
        return None


# Le due ROI sono mostrate in due output diversi, quindi possono usare lo stesso colore
# senza sovrapporsi visivamente.
ROI_DISPLAY_COLOR = "lime"
ROI_LINEWIDTH = 3.0


def draw_single_roi(ax, roi_data, label):
    """Disegna una sola ROI nel pannello corrente."""
    if not roi_data:
        return
    x, y, w, h = [int(roi_data[k]) for k in ["x", "y", "w", "h"]]
    rect = plt.Rectangle(
        (x, y),
        w,
        h,
        edgecolor=ROI_DISPLAY_COLOR,
        facecolor="none",
        lw=ROI_LINEWIDTH,
    )
    ax.add_patch(rect)
    ax.text(
        x,
        max(0, y - 5),
        label,
        color=ROI_DISPLAY_COLOR,
        fontsize=10,
        weight="bold",
        bbox=dict(facecolor="black", alpha=0.45, edgecolor="none", pad=1.8),
    )


def clamp_slider_to_video(video_id, slider_widget):
    """Aggiorna i limiti dello slider rispetto al frame di sincronizzazione del video."""
    frames = get_frames(video_id)
    if not frames:
        return False

    start_frame = get_sync_start(video_id)
    new_min = -int(start_frame)
    new_max = int(len(frames) - start_frame - 1)
    if new_max < new_min:
        new_max = new_min

    slider_widget.min = new_min
    slider_widget.max = new_max
    slider_widget.value = 0 if new_min <= 0 <= new_max else new_min
    return True


def render_roi_panel(video_id, slider_widget, output_widget, roi_file, roi_label, state_dict, panel_title):
    """Mostra un frame e una sola ROI: standard oppure 601, mai entrambe nello stesso output."""
    if not video_id:
        with output_widget:
            clear_output(wait=True)
            print(f"Nessun video disponibile per: {panel_title}")
        return

    frames = get_frames(video_id)
    if not frames:
        with output_widget:
            clear_output(wait=True)
            print(f"Nessun frame PNG trovato per il video {video_id}.")
        return

    offset = int(slider_widget.value)
    start_frame = get_sync_start(video_id)
    abs_idx = start_frame + offset
    abs_idx = max(0, min(abs_idx, len(frames) - 1))

    frame_name = frames[abs_idx]
    frame_path = os.path.join(FRAME_DIR, video_id, frame_name)

    img_gray = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        with output_widget:
            clear_output(wait=True)
            print(f"Impossibile leggere frame: {frame_path}")
        return

    metadata = get_thermal_metadata(video_id)
    if metadata:
        vmin = metadata.get("vmin_celsius", 0)
        vmax = metadata.get("vmax_celsius", 255)
        img_temp = vmin + (img_gray / 255.0) * (vmax - vmin)
        display_img = img_temp
        cmap_label = "Temperatura (C)"
    else:
        img_temp = None
        display_img = img_gray
        cmap_label = "Grigi (0-255)"

    roi_data = load_roi_file(roi_file)

    state_dict.clear()
    state_dict.update({
        "path": frame_path,
        "video_id": video_id,
        "offset": offset,
        "abs_idx": abs_idx,
        "metadata": metadata,
        "canonical_id": canonical_provino_id(video_id),
        "roi_file": roi_file,
        "roi_label": roi_label,
    })

    with output_widget:
        clear_output(wait=True)
        with plt.ioff():
            fig, ax = plt.subplots(figsize=(10, 6))
            im = ax.imshow(display_img, cmap="inferno")
            fig.colorbar(im, ax=ax, label=cmap_label)

            # Disegna solo la ROI del pannello corrente.
            # In questo modo la ROI del 601 non si sovrappone mai alla ROI standard.
            draw_single_roi(ax, roi_data, roi_label)

            title_parts = [
                panel_title,
                f"Video {video_id}",
                f"ID output {canonical_provino_id(video_id)}",
                f"t={offset}",
            ]
            if roi_data is None:
                title_parts.append(f"{roi_label} non ancora salvata")

            if img_temp is not None and roi_data is not None:
                x, y, w, h = [int(roi_data[k]) for k in ["x", "y", "w", "h"]]
                roi_crop = img_temp[y:y+h, x:x+w]
                if roi_crop.size:
                    title_parts.append(f"Media ROI: {np.mean(roi_crop):.1f} C")
                    title_parts.append(f"Max: {np.max(roi_crop):.1f} C")

            ax.set_title(" | ".join(title_parts))
            ax.axis("off")
            display(fig)
            plt.close(fig)


def select_roi_from_state(state_dict, target_file, target_label, force_standard_size=False):
    """Seleziona e salva la ROI usando esattamente lo stesso flusso OpenCV per standard e 601."""
    if "path" not in state_dict:
        print("\n[INFO] Prima visualizza un frame nel pannello corrispondente.")
        return

    frame_path = state_dict["path"]
    video_id = state_dict["video_id"]

    img_gray = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        print(f"\n[ERRORE] Impossibile leggere {frame_path}")
        return

    img_bgr = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
    win_name = f"Seleziona {target_label} - {video_id}"
    cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
    cv2.setWindowProperty(win_name, cv2.WND_PROP_TOPMOST, 1)

    print("\n=> [OpenCV] Trascina con il mouse per selezionare l'area.")
    print("=> Premi SPAZIO o INVIO per confermare.")
    roi = cv2.selectROI(win_name, img_bgr, showCrosshair=True, fromCenter=False)
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)

    x, y, w, h = [int(v) for v in roi]
    if w <= 0 or h <= 0:
        print("\n[INFO] Selezione ROI annullata.")
        return

    # Per il provino 601 la posizione e' libera, ma la dimensione viene allineata
    # alla ROI standard per mantenere compatibilita' con mean_model e calib_scores.
    if force_standard_size:
        roi_standard = load_roi_file(ROI_FILE)
        if roi_standard is not None:
            old_w, old_h = w, h
            w = int(roi_standard["w"])
            h = int(roi_standard["h"])
            print(f"[INFO] ROI 601: dimensione forzata a {w}x{h} px come la ROI standard.")
            print(f"       Mantengo l'angolo alto-sinistra selezionato ({x}, {y}); dimensioni selezionate ignorate: {old_w}x{old_h}.")
        else:
            print("[WARN] ROI standard non trovata: salvo la ROI 601 con le dimensioni selezionate.")

    H, W = img_gray.shape[:2]
    if x < 0 or y < 0 or x + w > W or y + h > H:
        print("[ERRORE] ROI fuori dai limiti del frame. Scegli un'area piu' interna.")
        print(f"Frame: {W}x{H} px | ROI: x={x}, y={y}, w={w}, h={h}")
        return

    roi_data = {
        "provino_id": canonical_provino_id(video_id),
        "source_video_id": video_id,
        "sync_offset_t": int(state_dict["offset"]),
        "abs_frame_idx": int(state_dict["abs_idx"]),
        "frame_path": frame_path,
        "x": int(x),
        "y": int(y),
        "w": int(w),
        "h": int(h),
        "display_color": ROI_DISPLAY_COLOR,
        "note": f"{target_label}. ROI visualizzata in pannello separato, senza sovrapposizione con altre ROI.",
    }

    with open(target_file, "w") as f:
        json.dump(roi_data, f, indent=4)

    print(f"[OK] {target_label} salvata ({x}, {y}, {w}, {h}) nel file '{target_file}'")

    # Aggiorna solo il pannello che ha generato la selezione.
    if target_file == ROI_FILE_601:
        render_roi_panel(video_601_dropdown.value, frame_601_slider, output_601_widget, ROI_FILE_601, "ROI 601", state_601, "Output 601")
    else:
        render_roi_panel(standard_video_dropdown.value, standard_frame_slider, output_standard_widget, ROI_FILE, "ROI standard", state_standard, "Output standard")


# ---------------------------------------------------------------------
# Due pannelli separati: uno per i provini standard, uno per il 601.
# La ROI viene selezionata nello stesso modo e mostrata nello stesso colore,
# ma non viene mai sovrapposta all'altra ROI nello stesso output video.
# ---------------------------------------------------------------------
standard_videos = [v for v in available_videos if not is_provino_601(v)]
video_601_options = [v for v in available_videos if is_provino_601(v)]

standard_video_dropdown = widgets.Dropdown(
    options=standard_videos,
    description="Video standard:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
standard_frame_slider = widgets.IntSlider(
    min=0,
    max=0,
    step=1,
    value=0,
    description="Frame standard:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
standard_roi_button = widgets.Button(
    description="Seleziona ROI standard",
    button_style="success",
    icon="crop",
    layout=widgets.Layout(width="240px"),
)
output_standard_widget = widgets.Output()
state_standard = {}

video_601_dropdown = widgets.Dropdown(
    options=video_601_options,
    description="Video 601:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
frame_601_slider = widgets.IntSlider(
    min=0,
    max=0,
    step=1,
    value=0,
    description="Frame 601:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)
roi_601_button = widgets.Button(
    description="Seleziona ROI 601",
    button_style="success",
    icon="crop",
    layout=widgets.Layout(width="240px"),
)
output_601_widget = widgets.Output()
state_601 = {}


def update_standard_panel(*args):
    video_id = standard_video_dropdown.value
    if not video_id:
        with output_standard_widget:
            clear_output(wait=True)
            print("Nessun video standard disponibile in dati/frame_thermo.")
        return
    if clamp_slider_to_video(video_id, standard_frame_slider):
        render_roi_panel(video_id, standard_frame_slider, output_standard_widget, ROI_FILE, "ROI standard", state_standard, "Output standard")


def update_601_panel(*args):
    video_id = video_601_dropdown.value
    if not video_id:
        with output_601_widget:
            clear_output(wait=True)
            print("Provino 601 non trovato in dati/frame_thermo.")
            print("Esegui prima estrazione_frame_601.ipynb sul file Rec-G3_S60.seq.")
        return
    if clamp_slider_to_video(video_id, frame_601_slider):
        render_roi_panel(video_id, frame_601_slider, output_601_widget, ROI_FILE_601, "ROI 601", state_601, "Output 601")


def show_standard_frame(*args):
    if standard_video_dropdown.value:
        render_roi_panel(standard_video_dropdown.value, standard_frame_slider, output_standard_widget, ROI_FILE, "ROI standard", state_standard, "Output standard")


def show_601_frame(*args):
    if video_601_dropdown.value:
        render_roi_panel(video_601_dropdown.value, frame_601_slider, output_601_widget, ROI_FILE_601, "ROI 601", state_601, "Output 601")


def on_select_standard_roi(_):
    select_roi_from_state(state_standard, ROI_FILE, "ROI standard", force_standard_size=False)


def on_select_601_roi(_):
    select_roi_from_state(state_601, ROI_FILE_601, "ROI 601", force_standard_size=True)


standard_video_dropdown.observe(update_standard_panel, names="value")
standard_frame_slider.observe(show_standard_frame, names="value")
standard_roi_button.on_click(on_select_standard_roi)

video_601_dropdown.observe(update_601_panel, names="value")
frame_601_slider.observe(show_601_frame, names="value")
roi_601_button.on_click(on_select_601_roi)

display(widgets.HTML("<h3>ROI standard - provini diversi da 601</h3>"))
display(widgets.HBox([standard_video_dropdown, standard_roi_button]))
display(standard_frame_slider)
display(output_standard_widget)

display(widgets.HTML("<h3>ROI dedicata - provino 601 / Rec-G3_S60</h3>"))
display(widgets.HBox([video_601_dropdown, roi_601_button]))
display(frame_601_slider)
display(output_601_widget)

update_standard_panel()
update_601_panel()



HTML(value='<h3>ROI standard - provini diversi da 601</h3>')

IntSlider(value=0, description='Frame standard:', layout=Layout(width='80%'), max=0, style=SliderStyle(descrip…

Output()

HTML(value='<h3>ROI dedicata - provino 601 / Rec-G3_S60</h3>')

IntSlider(value=0, description='Frame 601:', layout=Layout(width='80%'), max=0, style=SliderStyle(description_…

Output()

### 3. Estrazione Mini-Video (Passate dell'Ugello)

Questo codice itererà su tutti i provini per registrare i "mini-video" (ovvero tensori NumPy `[frames, height, width]`) della ROI impostata. 
Il processo:
1. Ignora i frame antecedenti allo `start` indicato in `map.json`.
2. Inizia la registrazione non appena l'ugello si muove verso **destra**.
3. Interrompe la registrazione solo se l'ugello inverte nettamente la marcia verso sinistra (oltre la `STOP_TOLERANCE`).
4. Salva il ritaglio termico, già convertito in gradi Celsius, sotto forma di `.npy` nella cartella `dati/mini_video_roi/`.

**Caso speciale provino 601.** Il video termico del provino 601 può comparire come `Rec-G3_S60` dopo l'estrazione dei frame. In questa versione viene trattato come provino `601` negli output, ma usa una ROI dedicata salvata in `dati/config/roi/roi_config_601.json`. La selezione ROI avviene nello stesso modo della ROI standard e viene mostrata con lo stesso colore, ma in un output video separato: la ROI del 601 non viene sovrapposta a video alla ROI degli altri provini. Per compatibilità con il modello conformal, la ROI del 601 può avere una posizione diversa ma deve mantenere la stessa dimensione della ROI standard. Le configurazioni ROI sono salvate in `dati/config/roi/`, non più nella root del progetto.


**Tracking termico mancante.** Se per il provino `601` non esiste ancora un file `trajectory_601.json`, questa versione prova a crearlo automaticamente dai frame termici usando lo stesso `kernel.png` dell'ugello. Il file `tracking_full_601.npy`, se presente, non viene usato: appartiene al tracking meccanico/allungamento e non contiene la traiettoria termica dell'ugello richiesta da questa cella.


In [18]:
import os
import json
import shutil
import cv2
import numpy as np
from tqdm.notebook import tqdm

# ==========================================
# PARAMETRI CONFIGURABILI DALL'UTENTE
# ==========================================
MAX_VIDEOS = 200           # Numero max di passate salvate per provino
MIN_FRAMES = 10            # Lunghezza minima in frame della passata
STOP_TOLERANCE = 5.0       # Tolleranza su X (in px) per decretare il movimento a sinistra (Stop)
START_TOLERANCE = 1.0      # Tolleranza su X (in px) per l'inizio del movimento a destra
SAVE_AS_NPY = True         # True = salva array NumPy; False = salva frame PNG
# ==========================================

# Nota importante:
# Questa cella richiede il tracking TERMICO dell'ugello, cioe' file trajectory_*.json.
# File come tracking_full_601.npy appartengono al tracking meccanico/allungamento e non sono
# compatibili con questa fase, perche' non contengono la posizione X dell'ugello sui frame termici.

TRACKING_DIR = os.path.join("dati", "tracking")
LEGACY_TRACKING_DIR = "dati"  # vecchio percorso: eventuali trajectory_*.json vengono migrati in TRACKING_DIR
KERNEL_PATH = os.path.join("dati", "kernel.png")
OUT_DIR = os.path.join("dati", "mini_video_roi")

os.makedirs(TRACKING_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


def load_required_roi(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} non trovata: {path}")
    with open(path, 'r') as f:
        return json.load(f)


# ROI standard: obbligatoria per tutti i provini tranne il 601.
roi_standard = load_required_roi(ROI_FILE, "ROI standard")
standard_w, standard_h = int(roi_standard['w']), int(roi_standard['h'])

# ROI dedicata al 601: se manca, il 601 viene saltato per evitare crop sbagliati.
roi_601 = None
if os.path.exists(ROI_FILE_601):
    roi_601 = load_required_roi(ROI_FILE_601, "ROI dedicata 601")
    if (int(roi_601['w']), int(roi_601['h'])) != (standard_w, standard_h):
        print("[WARN] La ROI 601 aveva dimensioni diverse dalla ROI standard.")
        print(f"       Correggo in memoria a {standard_w}x{standard_h} px per compatibilita' con il modello conformal.")
        roi_601['w'] = standard_w
        roi_601['h'] = standard_h
else:
    print(f"[INFO] {ROI_FILE_601} non trovato: il provino 601/Rec-G3_S60 verra' saltato finche' non salvi la ROI dedicata.")


with open(MAP_FILE, 'r') as f:
    sync_map = {str(item['id']).replace(".seq", ""): int(item['start']) for item in json.load(f)}


def list_processing_items():
    """
    Restituisce coppie (output_provino_id, source_video_id).
    Esempio: source Rec-G3_S60 -> output 601.
    """
    if not os.path.exists(FRAME_DIR):
        return []
    source_dirs = sorted([d for d in os.listdir(FRAME_DIR) if os.path.isdir(os.path.join(FRAME_DIR, d))])
    items = {}
    for source_id in source_dirs:
        output_id = canonical_provino_id(source_id)
        if output_id in items and items[output_id] != source_id:
            print(f"[WARN] Doppia sorgente per provino {output_id}: tengo {items[output_id]}, ignoro {source_id}")
            continue
        items[output_id] = source_id
    return [(out_id, src_id) for out_id, src_id in sorted(items.items(), key=lambda x: x[0])]


def tracking_name_candidates(output_id, source_id):
    candidates = [str(output_id), str(source_id), str(source_id).replace(".seq", "")]
    if output_id == SPECIAL_PROVINO_601:
        candidates.extend([a.replace(".seq", "") for a in SPECIAL_601_ALIASES])
    return list(dict.fromkeys([c for c in candidates if c]))


def target_tracking_file(output_id):
    """Percorso canonico unico per il tracking termico di ogni provino."""
    return os.path.join(TRACKING_DIR, f"trajectory_{output_id}.json")


def normalize_tracking_file(candidate_path, output_id):
    """
    Se trova un file trajectory_*.json in un percorso non standard, lo porta nel percorso canonico:
    dati/tracking/trajectory_<provino>.json.
    """
    target_path = target_tracking_file(output_id)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    if os.path.abspath(candidate_path) == os.path.abspath(target_path):
        return target_path

    if not os.path.exists(target_path):
        shutil.move(candidate_path, target_path)
        print(f"[INFO] Tracking termico del provino {output_id} normalizzato nel percorso standard.")
        return target_path

    print(f"[INFO] Tracking termico duplicato non standard ignorato per provino {output_id}.")
    return target_path


def find_existing_tracking_file(output_id, source_id):
    """
    Cerca il tracking termico e lo normalizza sempre nel percorso unico:
    dati/tracking/trajectory_<provino>.json.
    """
    target_path = target_tracking_file(output_id)
    if os.path.exists(target_path):
        return target_path

    # 1) Cerca eventuali file gia' in dati/tracking ma con nome alias.
    for candidate in tracking_name_candidates(output_id, source_id):
        p = os.path.join(TRACKING_DIR, f"trajectory_{candidate}.json")
        if os.path.exists(p):
            return normalize_tracking_file(p, output_id)

    # 2) Migrazione dal vecchio percorso dati/trajectory_*.json, se presente.
    for candidate in tracking_name_candidates(output_id, source_id):
        p = os.path.join(LEGACY_TRACKING_DIR, f"trajectory_{candidate}.json")
        if os.path.exists(p):
            return normalize_tracking_file(p, output_id)

    return None


def find_mechanical_tracking_full(output_id):
    """Trova eventuali tracking_full_*.npy solo per stampare un avviso: non sono usabili qui."""
    wanted = f"tracking_full_{output_id}.npy"
    found = []
    if os.path.exists("dati"):
        for root, _, files in os.walk("dati"):
            if wanted in files:
                found.append(os.path.join(root, wanted))
    return found


def create_thermal_trajectory(output_id, source_id, frames_dir):
    """
    Crea il tracking termico dell'ugello dai PNG in dati/frame_thermo/<source_id>/.
    Usa dati/kernel.png e salva sempre in dati/tracking/trajectory_<output_id>.json.
    """
    if not os.path.exists(KERNEL_PATH):
        print(f"[SKIP] Provino {output_id}: tracking termico mancante e kernel non trovato: {KERNEL_PATH}")
        return None

    kernel = cv2.imread(KERNEL_PATH, cv2.IMREAD_GRAYSCALE)
    if kernel is None:
        print(f"[SKIP] Provino {output_id}: impossibile leggere il kernel: {KERNEL_PATH}")
        return None

    h_k, w_k = kernel.shape[:2]
    frame_files = sorted([f for f in os.listdir(frames_dir) if f.lower().endswith('.png')])
    if not frame_files:
        print(f"[SKIP] Provino {output_id}: nessun PNG disponibile per creare il tracking termico.")
        return None

    print(f"[INFO] Tracking termico non trovato per provino {output_id} sorgente {source_id}.")
    print(f"       Lo creo ora con template matching su {len(frame_files)} frame usando {KERNEL_PATH}.")

    trajectory = []
    for idx, filename in enumerate(tqdm(frame_files, desc=f"Tracking termico {output_id}", leave=False)):
        path = os.path.join(frames_dir, filename)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        if img.shape[0] < h_k or img.shape[1] < w_k:
            print(f"[SKIP] Frame troppo piccolo per il kernel: {filename}")
            continue

        res = cv2.matchTemplate(img, kernel, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(res)
        cx = int(max_loc[0] + w_k // 2)
        cy = int(max_loc[1] + h_k // 2)
        trajectory.append({
            "frame_idx": int(idx),
            "filename": filename,
            "x": cx,
            "y": cy,
            "score": float(max_val),
        })

    if not trajectory:
        print(f"[SKIP] Provino {output_id}: tracking termico non creato, traiettoria vuota.")
        return None

    out_path = target_tracking_file(output_id)
    with open(out_path, 'w') as f:
        json.dump(trajectory, f, indent=4)

    scores = np.array([p['score'] for p in trajectory], dtype=float)
    print(f"[OK] Tracking termico creato per provino {output_id}.")
    print(f"     Score template matching: min={scores.min():.3f}, media={scores.mean():.3f}, max={scores.max():.3f}")
    return out_path


def get_or_create_tracking_file(output_id, source_id, frames_dir):
    tracking_file = find_existing_tracking_file(output_id, source_id)
    if tracking_file is not None:
        print(f"[OK] Tracking termico per provino {output_id}.")
        return tracking_file

    mechanical_files = find_mechanical_tracking_full(output_id)
    if mechanical_files:
        print(f"[INFO] Trovato tracking_full per provino {output_id}, ma non lo uso in conformal:")
        for p in mechanical_files:
            print(f"       - {p}")
        print("       Questo file appartiene al tracking meccanico/allungamento, non alla traiettoria termica dell'ugello.")

    return create_thermal_trajectory(output_id, source_id, frames_dir)


def roi_for_output_id(output_id):
    if output_id == SPECIAL_PROVINO_601:
        return roi_601
    return roi_standard


def validate_roi_inside_frame(roi_data, img_shape, output_id, source_id):
    H, W = img_shape[:2]
    rx, ry, rw, rh = [int(roi_data[k]) for k in ['x', 'y', 'w', 'h']]
    if rx < 0 or ry < 0 or rw <= 0 or rh <= 0 or rx + rw > W or ry + rh > H:
        print(f"[ERRORE] ROI fuori frame per provino {output_id} sorgente {source_id}.")
        print(f"         Frame: {W}x{H} px | ROI: x={rx}, y={ry}, w={rw}, h={rh}")
        return False
    return True


def save_mini_video(source_id, output_id, video_idx, filenames, out_dir, vmin_c, vmax_c, roi_data):
    rx, ry, rw, rh = [int(roi_data[k]) for k in ['x', 'y', 'w', 'h']]
    video_tensors = []
    for filename in filenames:
        path = os.path.join(FRAME_DIR, source_id, filename)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            crop = img[ry:ry+rh, rx:rx+rw]
            if crop.shape[:2] != (rh, rw):
                continue
            crop_celsius = vmin_c + (crop / 255.0) * (vmax_c - vmin_c)
            video_tensors.append(crop_celsius)

    if SAVE_AS_NPY and video_tensors:
        np.save(os.path.join(out_dir, f"stroke_{video_idx:03d}.npy"), np.array(video_tensors, dtype=np.float32))
    elif not SAVE_AS_NPY and video_tensors:
        stroke_dir = os.path.join(out_dir, f"stroke_{video_idx:03d}")
        os.makedirs(stroke_dir, exist_ok=True)
        for i, tensor in enumerate(video_tensors):
            norm_img = cv2.normalize(tensor, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            cv2.imwrite(os.path.join(stroke_dir, f"{i:03d}.png"), norm_img)


# 2. Ciclo principale di estrazione
processing_items = list_processing_items()
print("Provini/sorgenti da elaborare:")
for output_id, source_id in processing_items:
    extra = "  [ROI 601]" if output_id == SPECIAL_PROVINO_601 else ""
    print(f" - output {output_id} <- sorgente {source_id}{extra}")

for output_id, source_id in tqdm(processing_items, desc="Provini Elaborati"):
    roi_data = roi_for_output_id(output_id)
    if roi_data is None:
        print(f"[SKIP] Provino {output_id} sorgente {source_id}: ROI dedicata mancante ({ROI_FILE_601}).")
        continue

    start_frame = get_sync_start(source_id)
    frames_dir = os.path.join(FRAME_DIR, source_id)
    meta_file = os.path.join(frames_dir, "thermal_metadata.json")

    if not os.path.exists(meta_file):
        print(f"[SKIP] Provino {output_id} sorgente {source_id}: metadata termici non trovati.")
        continue

    frames = sorted([f for f in os.listdir(frames_dir) if f.lower().endswith('.png')])
    if not frames:
        print(f"[SKIP] Provino {output_id} sorgente {source_id}: nessun PNG trovato.")
        continue

    # Validazione ROI sul primo frame disponibile prima di processare tutto.
    first_img = cv2.imread(os.path.join(frames_dir, frames[0]), cv2.IMREAD_GRAYSCALE)
    if first_img is None or not validate_roi_inside_frame(roi_data, first_img.shape, output_id, source_id):
        continue

    tracking_file = get_or_create_tracking_file(output_id, source_id, frames_dir)
    if tracking_file is None:
        print(f"[SKIP] Provino {output_id} sorgente {source_id}: tracking termico non disponibile.")
        continue

    with open(tracking_file, 'r') as f:
        track_map = {item['filename']: item['x'] for item in json.load(f)}

    with open(meta_file, 'r') as f:
        meta = json.load(f)
    vmin_c, vmax_c = meta['vmin_celsius'], meta['vmax_celsius']

    if start_frame < len(frames):
        frames = frames[start_frame:]
    else:
        print(f"[SKIP] Provino {output_id} sorgente {source_id}: start_frame={start_frame} oltre il numero frame={len(frames)}.")
        continue

    out_provino_dir = os.path.join(OUT_DIR, output_id)
    os.makedirs(out_provino_dir, exist_ok=True)

    state = "IDLE"
    min_x = float('inf')
    max_x = 0
    current_video = []
    videos_saved = 0

    for filename in frames:
        if videos_saved >= MAX_VIDEOS:
            break

        x = track_map.get(filename, None)
        if x is None:
            continue

        if state == "IDLE":
            if x < min_x:
                min_x = x
            if x > min_x + START_TOLERANCE:
                state = "RECORDING"
                current_video = [filename]
                max_x = x

        elif state == "RECORDING":
            current_video.append(filename)
            if x > max_x:
                max_x = x

            if max_x - x > STOP_TOLERANCE:
                if len(current_video) >= MIN_FRAMES:
                    save_mini_video(source_id, output_id, videos_saved, current_video, out_provino_dir, vmin_c, vmax_c, roi_data)
                    videos_saved += 1
                state = "IDLE"
                current_video = []
                min_x = x

    print(f"Provino {output_id} (sorgente {source_id}): salvate {videos_saved} passate.")


Provini/sorgenti da elaborare:
 - output 101 <- sorgente 101
 - output 102 <- sorgente 102
 - output 301 <- sorgente 301
 - output 302 <- sorgente 302
 - output 303 <- sorgente 303
 - output 304 <- sorgente 304
 - output 601 <- sorgente 601  [ROI 601]
 - output 602 <- sorgente 602
 - output 901 <- sorgente 901
 - output 902 <- sorgente 902
 - output 992 <- sorgente 992


Provini Elaborati:   0%|          | 0/11 [00:00<?, ?it/s]

[OK] Tracking termico per provino 101.
Provino 101 (sorgente 101): salvate 200 passate.
[OK] Tracking termico per provino 102.
Provino 102 (sorgente 102): salvate 76 passate.
[OK] Tracking termico per provino 301.
Provino 301 (sorgente 301): salvate 75 passate.
[OK] Tracking termico per provino 302.
Provino 302 (sorgente 302): salvate 83 passate.
[OK] Tracking termico per provino 303.
Provino 303 (sorgente 303): salvate 76 passate.
[OK] Tracking termico per provino 304.
Provino 304 (sorgente 304): salvate 76 passate.
[OK] Tracking termico per provino 601.
Provino 601 (sorgente 601): salvate 87 passate.
[OK] Tracking termico per provino 602.
Provino 602 (sorgente 602): salvate 80 passate.
[OK] Tracking termico per provino 901.
Provino 901 (sorgente 901): salvate 82 passate.
[OK] Tracking termico per provino 902.
Provino 902 (sorgente 902): salvate 79 passate.
[OK] Tracking termico per provino 992.
Provino 992 (sorgente 992): salvate 159 passate.


### 4. Visualizzazione dei Segmenti Estratti

Usa questa interfaccia per esplorare i mini-video (le passate dell'ugello) appena generati. Seleziona il provino, la passata specifica (stroke) e muovi lo slider per scorrere i frame all'interno della ROI selezionata.

In [19]:
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

OUT_DIR = os.path.join("dati", "mini_video_roi")

if os.path.exists(OUT_DIR):
    avail_provini = sorted([d for d in os.listdir(OUT_DIR) if os.path.isdir(os.path.join(OUT_DIR, d))])
else:
    avail_provini = []

# Setup Widget
dropdown_provino = widgets.Dropdown(options=avail_provini, description='Provino:', style={'description_width': 'initial'})
dropdown_stroke = widgets.Dropdown(options=[], description='Passata:', style={'description_width': 'initial'})
slider_frame = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Frame interno:', layout=widgets.Layout(width='600px'), style={'description_width': 'initial'})
out_viewer = widgets.Output()

current_tensor = None

def update_strokes(*args):
    provino = dropdown_provino.value
    if not provino:
        dropdown_stroke.options = []
        return
    
    p_dir = os.path.join(OUT_DIR, provino)
    if os.path.exists(p_dir):
        strokes = sorted([f for f in os.listdir(p_dir) if f.endswith('.npy')])
        old_val = dropdown_stroke.value
        dropdown_stroke.options = strokes
        if strokes:
            new_val = strokes[0]
            dropdown_stroke.value = new_val
            if old_val == new_val:
                update_tensor()
    else:
        dropdown_stroke.options = []
        update_tensor()

def update_tensor(*args):
    global current_tensor
    provino = dropdown_provino.value
    stroke = dropdown_stroke.value
    
    if not provino or not stroke:
        current_tensor = None
        slider_frame.max = 0
        with out_viewer:
            clear_output(wait=True)
        return
        
    stroke_path = os.path.join(OUT_DIR, provino, stroke)
    try:
        current_tensor = np.load(stroke_path)
        old_val = slider_frame.value
        slider_frame.max = max(0, len(current_tensor) - 1)
        slider_frame.value = 0
        if old_val == 0:
            show_segment_frame()
    except Exception as e:
        current_tensor = None
        with out_viewer:
            clear_output(wait=True)
            print(f"Errore nel caricamento: {e}")

def show_segment_frame(*args):
    if current_tensor is None:
        return
        
    frame_idx = slider_frame.value
    img_celsius = current_tensor[frame_idx]
    
    with out_viewer:
        clear_output(wait=True)
        with plt.ioff():
            fig = plt.figure(figsize=(7, 5))
            im = plt.imshow(img_celsius, cmap='inferno')
            plt.colorbar(im, label='Temperatura (\u00b0C)')
            plt.title(
                f"Provino: {dropdown_provino.value} | File: {dropdown_stroke.value}\n"
                f"Frame della Passata: {frame_idx}/{len(current_tensor)-1} | Temp Media: {np.mean(img_celsius):.1f}\u00b0C"
            )
            plt.axis('off')
            display(fig)
            plt.close(fig)

# Collegamenti eventi
dropdown_provino.observe(update_strokes, names='value')
dropdown_stroke.observe(update_tensor, names='value')
slider_frame.observe(show_segment_frame, names='value')

# Mostra a schermo
display(widgets.HBox([dropdown_provino, dropdown_stroke]))
display(slider_frame)
display(out_viewer)

# Avvio iniziale
update_strokes()


IntSlider(value=0, description='Frame interno:', layout=Layout(width='600px'), max=0, style=SliderStyle(descri…

Output()

### 5. Allineamento temporale senza troncamento: ricampionamento in fase normalizzata

La versione originale del notebook calcolava una lunghezza target al percentile scelto e poi conservava solo i primi `L` frame di ogni passata. Questo rendeva i tensori confrontabili, ma eliminava informazione dalla parte finale dei mini-video.

In questa copia usiamo invece:

1. manteniamo i mini-video completi in `dati/mini_video_roi/` per l'analisi in **tempo reale**;
2. interpretiamo ogni mini-video come una funzione continua lungo la fase della passata, da 0% a 100%;
3. ricampioniamo ogni passata a una lunghezza standard `RESAMPLE_TARGET_L` usando interpolazione lineare;
4. salviamo i risultati in `dati/mini_video_resampled_phase/`;
5. usiamo questo nuovo dataset, non quello troncato, come input del modello conformal.

Nota concettuale: se una passata lunga viene portata a una lunghezza minore, non viene tagliata; viene **compressa temporalmente** sull'intera fase 0-100%. Se una passata corta viene portata a una lunghezza maggiore, viene **stirata/interpolata**. Questo non crea nuova informazione fisica, ma permette un confronto a pari numero di campioni senza buttare via la coda.


In [20]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ==========================================================
# METODO 3 + METODO 5
# Ricampionamento temporale su fase normalizzata 0-100%
# ==========================================================

RAW_ROI_DIR = Path("dati") / "mini_video_roi"                  # mini-video completi, lunghezza variabile
RESAMPLED_ROI_DIR = Path("dati") / "mini_video_resampled_phase" # mini-video ricampionati, lunghezza fissa
EXPORT_DIR_RESAMPLED = Path("dati") / "dataset_finale" / "conformal_resampled_phase"

EXPORT_DIR_RESAMPLED.mkdir(parents=True, exist_ok=True)

# Questa cartella diventa il nuovo input standard per split, modello conformal, inferenza e visualizzazioni conformal.
MODEL_INPUT_DIR = RESAMPLED_ROI_DIR
OUT_DIR = str(MODEL_INPUT_DIR)  # compatibilita' con le celle successive del notebook

# -----------------------------
# Parametri modificabili
# -----------------------------
# Strategie disponibili:
# - "manual": usa RESAMPLE_TARGET_L_MANUAL
# - "median": usa la mediana delle lunghezze originali
# - "mean": usa la media arrotondata delle lunghezze originali
# - "percentile": usa RESAMPLE_PERCENTILE delle lunghezze originali
# - "max": usa la lunghezza massima, quindi nessun video viene compresso, ma molti vengono stirati
RESAMPLE_TARGET_STRATEGY = "median"
RESAMPLE_TARGET_L_MANUAL = 24
RESAMPLE_PERCENTILE = 50

# Interpolazione lineare: buona scelta di partenza per curve termiche abbastanza regolari.
# Si puo' cambiare in "nearest" solo per test, ma non e' consigliato per curve T(t).
RESAMPLE_KIND = "linear"

# Evita residui di vecchie esecuzioni nella nuova cartella ricampionata.
OVERWRITE_RESAMPLED = True
MIN_FRAMES_FOR_RESAMPLING = 2


def collect_video_lengths(in_dir: Path) -> pd.DataFrame:
    """Raccoglie lunghezze e shape di tutti gli stroke .npy nel dataset completo."""
    rows = []
    if not in_dir.exists():
        raise FileNotFoundError(f"Cartella non trovata: {in_dir}")

    provini = sorted([p for p in in_dir.iterdir() if p.is_dir()])
    for p_dir in provini:
        for stroke_path in sorted(p_dir.glob("*.npy")):
            arr = np.load(stroke_path, mmap_mode="r")
            rows.append({
                "provino_id": p_dir.name,
                "stroke_file": stroke_path.name,
                "path": str(stroke_path),
                "original_length": int(arr.shape[0]),
                "height": int(arr.shape[1]),
                "width": int(arr.shape[2]),
            })
    return pd.DataFrame(rows)


def choose_target_length(lengths, strategy="manual", manual_value=24, percentile=50) -> int:
    """Sceglie la lunghezza comune del nuovo dataset ricampionato."""
    lengths = np.asarray(lengths, dtype=float)
    if lengths.size == 0:
        raise ValueError("Nessuna lunghezza disponibile per scegliere RESAMPLE_TARGET_L.")

    strategy = strategy.lower()
    if strategy == "manual":
        target = int(manual_value)
    elif strategy == "median":
        target = int(round(float(np.median(lengths))))
    elif strategy == "mean":
        target = int(round(float(np.mean(lengths))))
    elif strategy == "percentile":
        target = int(round(float(np.percentile(lengths, percentile))))
    elif strategy == "max":
        target = int(np.max(lengths))
    else:
        raise ValueError(f"Strategia non riconosciuta: {strategy}")

    if target < MIN_FRAMES_FOR_RESAMPLING:
        target = MIN_FRAMES_FOR_RESAMPLING
    return target


def resample_video_phase(video: np.ndarray, target_L: int, kind: str = "linear") -> np.ndarray:
    """
    Ricampiona un mini-video [L, H, W] su una griglia di fase comune [target_L, H, W].

    La fase 0.0 corrisponde al primo frame della passata.
    La fase 1.0 corrisponde all'ultimo frame della passata.
    Quindi l'intera passata viene sempre usata: non c'e' troncamento.
    """
    video = np.asarray(video, dtype=np.float32)
    original_L = int(video.shape[0])

    if original_L < MIN_FRAMES_FOR_RESAMPLING:
        raise ValueError(f"Video troppo corto per interpolare: L={original_L}")

    if original_L == target_L:
        return video.copy()

    old_phase = np.linspace(0.0, 1.0, original_L, dtype=np.float32)
    new_phase = np.linspace(0.0, 1.0, target_L, dtype=np.float32)

    if kind == "nearest":
        nearest_idx = np.searchsorted(old_phase, new_phase, side="left")
        nearest_idx = np.clip(nearest_idx, 0, original_L - 1)
        return video[nearest_idx].astype(np.float32)

    if kind != "linear":
        raise ValueError("Per ora sono supportati solo kind='linear' e kind='nearest'.")

    # Uso scipy se disponibile: interpola lungo l'asse temporale senza mischiare i pixel spaziali.
    try:
        from scipy.interpolate import interp1d
        f = interp1d(
            old_phase,
            video,
            axis=0,
            kind="linear",
            bounds_error=False,
            fill_value="extrapolate",
            assume_sorted=True,
        )
        return f(new_phase).astype(np.float32)
    except Exception:
        # Fallback senza scipy: interpola ogni pixel dopo reshape [L, H*W].
        flat = video.reshape(original_L, -1)
        out_flat = np.empty((target_L, flat.shape[1]), dtype=np.float32)
        for col in range(flat.shape[1]):
            out_flat[:, col] = np.interp(new_phase, old_phase, flat[:, col]).astype(np.float32)
        return out_flat.reshape((target_L,) + video.shape[1:]).astype(np.float32)


# 1. Inventario dataset completo
length_df = collect_video_lengths(RAW_ROI_DIR)

if length_df.empty:
    raise RuntimeError(f"Nessun file .npy trovato in {RAW_ROI_DIR}")

shape_counts = length_df.groupby(["height", "width"]).size().reset_index(name="count")
print("Shape ROI trovate:")
print(shape_counts.to_string(index=False))

lengths = length_df["original_length"].to_numpy()
RESAMPLE_TARGET_L = choose_target_length(
    lengths,
    strategy=RESAMPLE_TARGET_STRATEGY,
    manual_value=RESAMPLE_TARGET_L_MANUAL,
    percentile=RESAMPLE_PERCENTILE,
)

print("\nDistribuzione lunghezze originali:")
print(f"  Min:    {int(np.min(lengths))}")
print(f"  Max:    {int(np.max(lengths))}")
print(f"  Media:  {float(np.mean(lengths)):.2f}")
print(f"  Mediana:{float(np.median(lengths)):.2f}")
print(f"  P15:    {float(np.percentile(lengths, 15)):.2f}")
print(f"  P50:    {float(np.percentile(lengths, 50)):.2f}")
print(f"  P75:    {float(np.percentile(lengths, 75)):.2f}")
print(f"\nRESAMPLE_TARGET_L scelto: {RESAMPLE_TARGET_L} frame")
print(f"Strategia: {RESAMPLE_TARGET_STRATEGY}")

# 2. Prepara cartella output pulita
if OVERWRITE_RESAMPLED and RESAMPLED_ROI_DIR.exists():
    shutil.rmtree(RESAMPLED_ROI_DIR)
RESAMPLED_ROI_DIR.mkdir(parents=True, exist_ok=True)

# 3. Ricampionamento vero e proprio
metadata_rows = []
errors = []

for row in tqdm(length_df.itertuples(index=False), total=len(length_df), desc="Ricampionamento fase"):
    provino_id = str(row.provino_id)
    stroke_file = str(row.stroke_file)
    in_path = Path(row.path)
    out_dir = RESAMPLED_ROI_DIR / provino_id
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / stroke_file

    try:
        video = np.load(in_path).astype(np.float32)
        original_L = int(video.shape[0])
        resampled = resample_video_phase(video, RESAMPLE_TARGET_L, kind=RESAMPLE_KIND)
        np.save(out_path, resampled.astype(np.float32))

        metadata_rows.append({
            "provino_id": provino_id,
            "stroke_file": stroke_file,
            "source_path": str(in_path),
            "resampled_path": str(out_path),
            "original_length": original_L,
            "target_length": int(RESAMPLE_TARGET_L),
            "height": int(video.shape[1]),
            "width": int(video.shape[2]),
            "method": "phase_resample_linear" if RESAMPLE_KIND == "linear" else f"phase_resample_{RESAMPLE_KIND}",
            "compression_factor": float(original_L / RESAMPLE_TARGET_L),
            "was_compressed": bool(original_L > RESAMPLE_TARGET_L),
            "was_stretched": bool(original_L < RESAMPLE_TARGET_L),
            "lost_frames_by_truncation": 0,
        })
    except Exception as exc:
        errors.append({
            "provino_id": provino_id,
            "stroke_file": stroke_file,
            "error": repr(exc),
        })

resampling_metadata = pd.DataFrame(metadata_rows)
resampling_errors = pd.DataFrame(errors)

# 4. Salvataggio metadata di tracciabilita'
metadata_path = RESAMPLED_ROI_DIR / "resampling_metadata.csv"
resampling_metadata.to_csv(metadata_path, index=False)
resampling_metadata.to_csv(EXPORT_DIR_RESAMPLED / "resampling_metadata.csv", index=False)

config = {
    "raw_roi_dir": str(RAW_ROI_DIR),
    "resampled_roi_dir": str(RESAMPLED_ROI_DIR),
    "model_input_dir": str(MODEL_INPUT_DIR),
    "resample_target_L": int(RESAMPLE_TARGET_L),
    "resample_target_strategy": RESAMPLE_TARGET_STRATEGY,
    "resample_target_L_manual": int(RESAMPLE_TARGET_L_MANUAL),
    "resample_percentile": float(RESAMPLE_PERCENTILE),
    "resample_kind": RESAMPLE_KIND,
    "note": "Ogni stroke e' ricampionato sull'intera fase 0-100%; nessun frame finale e' eliminato per troncamento.",
}
with open(RESAMPLED_ROI_DIR / "resampling_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)
with open(EXPORT_DIR_RESAMPLED / "resampling_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

if not resampling_errors.empty:
    resampling_errors.to_csv(RESAMPLED_ROI_DIR / "resampling_errors.csv", index=False)
    print("\n[ATTENZIONE] Alcuni stroke non sono stati ricampionati:")
    print(resampling_errors.head(20).to_string(index=False))

# 5. Report finale
print("\nRicampionamento completato.")
print(f"Video ricampionati: {len(resampling_metadata)}")
print(f"Errori: {len(resampling_errors)}")
print(f"Cartella output: {RESAMPLED_ROI_DIR}")
print(f"Metadata: {metadata_path}")

print("\nSintesi compressione/stiramento:")
print(resampling_metadata[["original_length", "target_length", "compression_factor", "was_compressed", "was_stretched"]].describe().to_string())

print("\nConteggio per provino:")
print(resampling_metadata.groupby("provino_id").size().rename("n_strokes").to_string())


Shape ROI trovate:
 height  width  count
     30     68   1073

Distribuzione lunghezze originali:
  Min:    10
  Max:    43
  Media:  15.58
  Mediana:16.00
  P15:    12.00
  P50:    16.00
  P75:    18.00

RESAMPLE_TARGET_L scelto: 16 frame
Strategia: median


Ricampionamento fase:   0%|          | 0/1073 [00:00<?, ?it/s]


Ricampionamento completato.
Video ricampionati: 1073
Errori: 0
Cartella output: dati\mini_video_resampled_phase
Metadata: dati\mini_video_resampled_phase\resampling_metadata.csv

Sintesi compressione/stiramento:
       original_length  target_length  compression_factor
count      1073.000000         1073.0         1073.000000
mean         15.579683           16.0            0.973730
std           3.067560            0.0            0.191722
min          10.000000           16.0            0.625000
25%          13.000000           16.0            0.812500
50%          16.000000           16.0            1.000000
75%          18.000000           16.0            1.125000
max          43.000000           16.0            2.687500

Conteggio per provino:
provino_id
101    200
102     76
301     75
302     83
303     76
304     76
601     87
602     80
901     82
902     79
992    159


### 6. Funzione Temperatura nel Tempo: tempo reale completo oppure fase normalizzata

Questa cella distingue due viste diverse dello stesso fenomeno:

- **Tempo reale completo**: legge da `dati/mini_video_roi/`. La curva `T(t)` usa tutti i frame originali della passata e conserva la durata reale.
- **Fase normalizzata ricampionata**: legge da `dati/mini_video_resampled_phase/`. La curva usa sempre `RESAMPLE_TARGET_L` punti distribuiti tra 0% e 100% della passata.

La seconda vista è quella usata dal modello conformal perché produce tensori tutti della stessa shape, ma la prima resta disponibile per verificare che la coda termica non sia sparita.


In [21]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Cartelle dati: raw = tempo reale; resampled = fase normalizzata
try:
    RAW_ROI_DIR
except NameError:
    RAW_ROI_DIR = Path("dati") / "mini_video_roi"
try:
    RESAMPLED_ROI_DIR
except NameError:
    RESAMPLED_ROI_DIR = Path("dati") / "mini_video_resampled_phase"

source_dropdown = widgets.Dropdown(
    options=[
        ("Tempo reale completo - dati originali", "raw"),
        ("Fase normalizzata ricampionata - input modello", "resampled"),
    ],
    value="raw",
    description="Sorgente:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
dropdown_prov = widgets.Dropdown(description="Provino:", style={"description_width": "initial"})
dropdown_strk = widgets.Dropdown(description="Passata:", style={"description_width": "initial"})
slider_x = widgets.IntSlider(description="Pixel X:", style={"description_width": "initial"})
slider_y = widgets.IntSlider(description="Pixel Y:", style={"description_width": "initial"})
out_func = widgets.Output()


def current_data_dir() -> Path:
    return RAW_ROI_DIR if source_dropdown.value == "raw" else RESAMPLED_ROI_DIR


def list_provini(data_dir: Path):
    if not data_dir.exists():
        return []
    return sorted([p.name for p in data_dir.iterdir() if p.is_dir()])


def list_strokes(data_dir: Path, provino_id: str):
    p_dir = data_dir / str(provino_id)
    if not p_dir.exists():
        return []
    return sorted([p.name for p in p_dir.glob("*.npy")])


def init_ui(*args):
    data_dir = current_data_dir()
    provs = list_provini(data_dir)
    dropdown_prov.options = provs
    if provs:
        dropdown_prov.value = provs[0]
        update_strokes_list()
    else:
        dropdown_strk.options = []
        with out_func:
            clear_output(wait=True)
            print(f"Nessun dato trovato in: {data_dir}")


def update_strokes_list(*args):
    data_dir = current_data_dir()
    p = dropdown_prov.value
    if not p:
        return
    strokes = list_strokes(data_dir, p)

    old_val = dropdown_strk.value
    dropdown_strk.options = strokes
    if strokes:
        new_val = strokes[0]
        dropdown_strk.value = new_val
        sample = np.load(data_dir / p / new_val)
        slider_y.max = sample.shape[1] - 1
        slider_x.max = sample.shape[2] - 1
        if old_val == new_val:
            update_plot()
    else:
        with out_func:
            clear_output(wait=True)
            print(f"Nessuno stroke trovato per provino {p} in {data_dir}")


def update_plot(*args):
    data_dir = current_data_dir()
    p = dropdown_prov.value
    s = dropdown_strk.value
    if not p or not s:
        return

    x, y = slider_x.value, slider_y.value
    video = np.load(data_dir / p / s)
    temp_func = video[:, y, x]

    if source_dropdown.value == "raw":
        x_axis = np.arange(video.shape[0])
        x_label = "Frame reale della passata"
        title_suffix = "tempo reale completo"
    else:
        x_axis = np.linspace(0, 100, video.shape[0])
        x_label = "Fase normalizzata della passata (%)"
        title_suffix = "fase ricampionata"

    with out_func:
        out_func.clear_output(wait=True)
        with plt.ioff():
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

            mean_img = np.mean(video, axis=0)
            im = ax1.imshow(mean_img, cmap="inferno")
            ax1.plot(x, y, "ro", markersize=8)
            ax1.set_title(f"ROI media - riferimento pixel (x={x}, y={y})")
            plt.colorbar(im, ax=ax1, label="Temperatura media (°C)")

            ax2.plot(x_axis, temp_func, linewidth=2)
            ax2.set_title(f"T(t) | {p} - {s} | {title_suffix}\nL = {video.shape[0]} campioni")
            ax2.set_xlabel(x_label)
            ax2.set_ylabel("Temperatura (°C)")
            ax2.grid(True)

            plt.tight_layout()
            display(fig)
            plt.close(fig)


source_dropdown.observe(init_ui, names="value")
dropdown_prov.observe(update_strokes_list, names="value")
dropdown_strk.observe(update_plot, names="value")
slider_x.observe(update_plot, names="value")
slider_y.observe(update_plot, names="value")

display(source_dropdown)
display(widgets.HBox([dropdown_prov, dropdown_strk]))
display(widgets.HBox([slider_x, slider_y]))
display(out_func)

init_ui()


Dropdown(description='Sorgente:', layout=Layout(width='420px'), options=(('Tempo reale completo - dati origina…

Output()

### 7. Calcolo e visualizzazione delle funzioni mediate: tempo reale e fase normalizzata

La media viene calcolata in due modi:

1. **Media in tempo reale completo**: usa gli stroke originali a lunghezza variabile. Per ogni frame reale `t`, la media include solo gli stroke che possiedono davvero quel frame. La cella mostra anche `n_valid(t)`, cioè quante passate contribuiscono a ogni istante.
2. **Media in fase normalizzata**: usa il dataset ricampionato a lunghezza fissa. Ogni passata contribuisce da 0% a 100%, quindi non perdiamo la parte finale per troncamento.

Questa distinzione serve a non confondere il tempo fisico reale con la fase normalizzata della passata.


In [22]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    RAW_ROI_DIR
except NameError:
    RAW_ROI_DIR = Path("dati") / "mini_video_roi"
try:
    RESAMPLED_ROI_DIR
except NameError:
    RESAMPLED_ROI_DIR = Path("dati") / "mini_video_resampled_phase"


def load_videos_from_folder(folder: Path):
    strokes = sorted(folder.glob("*.npy"))
    return [np.load(p).astype(np.float32) for p in strokes], [p.name for p in strokes]


def mean_variable_length(videos):
    """
    Media in tempo reale senza padding a zero.
    Per ogni t, somma solo i video che hanno davvero il frame t.
    Restituisce:
    - mean_video: [max_L, H, W]
    - n_valid: [max_L], numero di passate disponibili a ogni frame reale
    """
    if not videos:
        return None, None

    max_L = max(v.shape[0] for v in videos)
    H, W = videos[0].shape[1], videos[0].shape[2]
    acc = np.zeros((max_L, H, W), dtype=np.float64)
    n_valid = np.zeros(max_L, dtype=np.int32)

    for v in videos:
        L = v.shape[0]
        acc[:L] += v
        n_valid[:L] += 1

    mean_video = np.full((max_L, H, W), np.nan, dtype=np.float32)
    valid = n_valid > 0
    mean_video[valid] = (acc[valid] / n_valid[valid, None, None]).astype(np.float32)
    return mean_video, n_valid


def mean_fixed_length(videos):
    if not videos:
        return None
    return np.mean(np.stack(videos, axis=0), axis=0).astype(np.float32)


mean_tensors_realtime = {}
valid_counts_realtime = {}
mean_tensors_phase = {}

print("Calcolo medie in tempo reale e in fase normalizzata...")

# 1. Medie tempo reale completo
if RAW_ROI_DIR.exists():
    for p_dir in sorted([p for p in RAW_ROI_DIR.iterdir() if p.is_dir()]):
        videos, _ = load_videos_from_folder(p_dir)
        if not videos:
            continue
        mean_video, n_valid = mean_variable_length(videos)
        mean_tensors_realtime[p_dir.name] = mean_video
        valid_counts_realtime[p_dir.name] = n_valid

# 2. Medie fase normalizzata ricampionata
if RESAMPLED_ROI_DIR.exists():
    for p_dir in sorted([p for p in RESAMPLED_ROI_DIR.iterdir() if p.is_dir()]):
        videos, _ = load_videos_from_folder(p_dir)
        if not videos:
            continue
        mean_tensors_phase[p_dir.name] = mean_fixed_length(videos)

print("Calcolo terminato.")
print(f"Provini con media tempo reale: {len(mean_tensors_realtime)}")
print(f"Provini con media fase normalizzata: {len(mean_tensors_phase)}")

# Mantengo mean_tensors come alias della versione usata dal modello per compatibilita' con eventuali celle successive.
mean_tensors = mean_tensors_phase

mode_dropdown = widgets.Dropdown(
    options=[
        ("Tempo reale completo", "realtime"),
        ("Fase normalizzata ricampionata", "phase"),
    ],
    value="phase",
    description="Media:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px"),
)
drop_p = widgets.Dropdown(description="Provino:", style={"description_width": "initial"})
sl_x = widgets.IntSlider(description="Pixel X:", style={"description_width": "initial"})
sl_y = widgets.IntSlider(description="Pixel Y:", style={"description_width": "initial"})
out_mean = widgets.Output()


def current_mean_dict():
    return mean_tensors_realtime if mode_dropdown.value == "realtime" else mean_tensors_phase


def init_mean_ui(*args):
    data = current_mean_dict()
    provs = sorted(data.keys())
    drop_p.options = provs
    if not provs:
        with out_mean:
            clear_output(wait=True)
            print("Nessuna media disponibile per questa modalita'.")
        return
    drop_p.value = provs[0]
    sample = data[provs[0]]
    sl_y.max = sample.shape[1] - 1
    sl_x.max = sample.shape[2] - 1
    update_mean_plot()


def update_mean_plot(*args):
    data = current_mean_dict()
    p = drop_p.value
    if not p or p not in data:
        return

    x, y = sl_x.value, sl_y.value
    mean_video = data[p]
    temp_func = mean_video[:, y, x]

    if mode_dropdown.value == "realtime":
        x_axis = np.arange(mean_video.shape[0])
        x_label = "Frame reale della passata"
        title = f"T(t) MEDIA in tempo reale | Provino {p}"
        n_valid = valid_counts_realtime[p]
    else:
        x_axis = np.linspace(0, 100, mean_video.shape[0])
        x_label = "Fase normalizzata della passata (%)"
        title = f"T(fase) MEDIA ricampionata | Provino {p}"
        n_valid = None

    with out_mean:
        out_mean.clear_output(wait=True)
        with plt.ioff():
            if mode_dropdown.value == "realtime":
                fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
            else:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
                ax3 = None

            mean_img = np.nanmean(mean_video, axis=0)
            im = ax1.imshow(mean_img, cmap="inferno")
            ax1.plot(x, y, "ro", markersize=8)
            ax1.set_title(f"ROI media generale - (x={x}, y={y})")
            plt.colorbar(im, ax=ax1, label="Temperatura media (°C)")

            ax2.plot(x_axis, temp_func, linewidth=3)
            ax2.set_title(title)
            ax2.set_xlabel(x_label)
            ax2.set_ylabel("Temperatura media (°C)")
            ax2.grid(True)

            if ax3 is not None:
                ax3.plot(x_axis, n_valid, linewidth=2)
                ax3.set_title("Passate che contribuiscono alla media")
                ax3.set_xlabel("Frame reale della passata")
                ax3.set_ylabel("n_valid(t)")
                ax3.grid(True)

            plt.tight_layout()
            display(fig)
            plt.close(fig)


mode_dropdown.observe(init_mean_ui, names="value")
drop_p.observe(update_mean_plot, names="value")
sl_x.observe(update_mean_plot, names="value")
sl_y.observe(update_mean_plot, names="value")

display(mode_dropdown)
display(drop_p)
display(widgets.HBox([sl_x, sl_y]))
display(out_mean)

init_mean_ui()


Calcolo medie in tempo reale e in fase normalizzata...
Calcolo terminato.
Provini con media tempo reale: 11
Provini con media fase normalizzata: 11


Dropdown(description='Media:', index=1, layout=Layout(width='360px'), options=(('Tempo reale completo', 'realt…

Dropdown(description='Provino:', options=(), style=DescriptionStyle(description_width='initial'), value=None)

Output()

### 8. Split dei Dati (Provino 992) sul dataset ricampionato

Da qui in avanti il modello conformal lavora su `MODEL_INPUT_DIR`, cioè `dati/mini_video_resampled_phase/`. Tutti gli stroke hanno la stessa lunghezza `RESAMPLE_TARGET_L`, ma ogni stroke è stato ottenuto usando l'intera passata 0-100%, non tagliando i primi frame.


In [23]:
import numpy as np
import os
from pathlib import Path

# Impostazioni Split
PROVINO_RIF = "992"

try:
    MODEL_INPUT_DIR
except NameError:
    MODEL_INPUT_DIR = Path("dati") / "mini_video_resampled_phase"

OUT_DIR = str(MODEL_INPUT_DIR)  # usato dalle celle successive

# Parametri modificabili
TRAIN_PCT = 0.5
CALIB_PCT = 0.3
TEST_PCT = 0.2
RANDOMIZE = True
SEED = 42

rif_dir = Path(OUT_DIR) / PROVINO_RIF
strokes = []
if rif_dir.exists():
    strokes = sorted([f.name for f in rif_dir.glob("*.npy")])

print(f"Dataset modello: {OUT_DIR}")
print(f"Trovati {len(strokes)} mini-video per il provino {PROVINO_RIF}")

indices = np.arange(len(strokes))
if RANDOMIZE:
    np.random.seed(SEED)
    np.random.shuffle(indices)

n_train = int(len(strokes) * TRAIN_PCT)
n_calib = int(len(strokes) * CALIB_PCT)

train_idx = indices[:n_train]
calib_idx = indices[n_train:n_train+n_calib]
test_idx = indices[n_train+n_calib:]

train_strokes = [strokes[i] for i in train_idx]
calib_strokes = [strokes[i] for i in calib_idx]
test_strokes = [strokes[i] for i in test_idx]

print(f"Train: {len(train_strokes)} | Calib: {len(calib_strokes)} | Test: {len(test_strokes)}")


def load_tensors(stroke_list, p_id):
    if not stroke_list:
        return np.array([])
    arrays = [np.load(Path(OUT_DIR) / p_id / s).astype(np.float32) for s in stroke_list]
    shapes = {arr.shape for arr in arrays}
    if len(shapes) != 1:
        raise ValueError(f"Shape non uniformi nel dataset modello: {shapes}")
    return np.stack(arrays, axis=0)

train_data = load_tensors(train_strokes, PROVINO_RIF)
calib_data = load_tensors(calib_strokes, PROVINO_RIF)
test_data = load_tensors(test_strokes, PROVINO_RIF)

if len(train_data) > 0:
    print("Shape train_data:", train_data.shape)
if len(calib_data) > 0:
    print("Shape calib_data:", calib_data.shape)
if len(test_data) > 0:
    print("Shape test_data:", test_data.shape)


Dataset modello: dati\mini_video_resampled_phase
Trovati 159 mini-video per il provino 992
Train: 79 | Calib: 47 | Test: 33
Shape train_data: (79, 16, 30, 68)
Shape calib_data: (47, 16, 30, 68)
Shape test_data: (33, 16, 30, 68)


### 9. Creazione del Modello Conformal e Calibrazione

Il modello è ancora definito come media dei frame di training. La differenza rispetto alla versione originale è che ora i frame non provengono da un troncamento ai primi `L` campioni, ma dal dataset ricampionato sull'intera fase della passata.


In [24]:
# 1. Modello: media sul set di Train per ogni fase/frame, Y, X
# train_data shape: [N_passate, RESAMPLE_TARGET_L, H, W]
if len(train_data) > 0:
    mean_model = np.mean(train_data, axis=0).astype(np.float32)  # shape: [Frames, H, W]
    print("Modello medio calcolato con successo.")
    print("Shape mean_model:", mean_model.shape)
else:
    print("Errore: nessun dato di training.")
    mean_model = None

# 2. Calibrazione: score di non-conformita' = distanza assoluta dal modello medio
if mean_model is not None and len(calib_data) > 0:
    calib_scores = np.abs(calib_data - mean_model).astype(np.float32)
    print("Punteggi di calibrazione calcolati e salvati in 'calib_scores'.")
    print("Shape calib_scores:", calib_scores.shape)
else:
    print("Errore: nessun dato di calibrazione.")
    calib_scores = np.array([])


Modello medio calcolato con successo.
Shape mean_model: (16, 30, 68)
Punteggi di calibrazione calcolati e salvati in 'calib_scores'.
Shape calib_scores: (47, 16, 30, 68)


### 10. Inferenza Globale su Tutti i Provini

Applichiamo il modello conformal a tutti gli stroke ricampionati in `MODEL_INPUT_DIR`. Tutti i tensori dovrebbero avere la stessa shape, perché derivano dal ricampionamento in fase normalizzata.


In [25]:
from pathlib import Path
import numpy as np

try:
    MODEL_INPUT_DIR
except NameError:
    MODEL_INPUT_DIR = Path("dati") / "mini_video_resampled_phase"

OUT_DIR = str(MODEL_INPUT_DIR)
inference_scores = {}  # dict: provino -> stroke -> score_tensor
skipped = []

provini = sorted([d.name for d in Path(OUT_DIR).iterdir() if d.is_dir()]) if Path(OUT_DIR).exists() else []

for p in provini:
    inference_scores[p] = {}
    p_dir = Path(OUT_DIR) / p
    p_strokes = sorted([f.name for f in p_dir.glob("*.npy")])
    for s in p_strokes:
        if mean_model is None:
            continue
        video = np.load(p_dir / s).astype(np.float32)
        if video.shape == mean_model.shape:
            inference_scores[p][s] = np.abs(video - mean_model).astype(np.float32)
        else:
            skipped.append((p, s, video.shape, mean_model.shape))

print(f"Inferenza completata su {len(provini)} provini.")
print(f"Stroke con score calcolato: {sum(len(v) for v in inference_scores.values())}")
print(f"Stroke saltati per shape non coerente: {len(skipped)}")
if skipped[:10]:
    print("Esempi saltati:")
    for item in skipped[:10]:
        print(item)


Inferenza completata su 11 provini.
Stroke con score calcolato: 1073
Stroke saltati per shape non coerente: 0


### 11. Visualizzazione binaria delle anomalie: pixel anomalo / non anomalo

Questa cella mostra la decisione conformal nella forma più diretta: per ogni pixel indica se è anomalo oppure no.

La regola usata è la stessa logica conformal già usata dal notebook per contare le anomalie:

- lo score di non-conformità del pixel è `abs(video - mean_model)`, già presente in `inference_scores`;
- la soglia viene stimata pixel-per-pixel dagli score di calibrazione `calib_scores`;
- un pixel viene marcato come anomalo se il suo score supera il quantile conformal calibrato.

La visualizzazione è stata ridotta a **due pannelli**:

1. **Frame termico originale**, per mantenere il riferimento fisico del dato;
2. **Overlay binario rosso**, dove i pixel rossi sono quelli classificati come anomali.

Il valore predefinito è `alpha = 0.10`, coerente con la feature termica usata nella fase finale del progetto (`p = 0.10`). Lo slider consente comunque di cambiare alpha.


In [27]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================================
# Visualizzazione binaria delle anomalie conformal, con 2 pannelli
# ==========================================================
# Questa cella NON mostra quanto e' anomalo un pixel.
# Mostra solo se il pixel e' anomalo oppure no, usando un overlay rosso ben visibile.
#
# Regola decisionale, coerente con il resto del notebook:
# 1. score_test = abs(frame_test - mean_model)
# 2. threshold = quantile conformal degli score di calibrazione, pixel per pixel
# 3. pixel anomalo se score_test > threshold
#
# Default alpha = 0.10:
# - e' una soglia conformal interpretabile;
# - e' coerente con l'analisi finale del progetto basata su p = 0.10;
# - e' meno severa di 0.05, utile quando l'obiettivo e' evidenziare zone termiche sospette.

DEFAULT_ALPHA_BINARY = 0.10

try:
    MODEL_INPUT_DIR
except NameError:
    MODEL_INPUT_DIR = Path("dati") / "mini_video_resampled_phase"

OUT_DIR = str(MODEL_INPUT_DIR)


def conformal_quantile_threshold(calib_scores_frame: np.ndarray, alpha: float) -> tuple[np.ndarray, float]:
    """
    Calcola la soglia conformal pixel-per-pixel per un frame/fase.

    Parameters
    ----------
    calib_scores_frame:
        Array [N_calib, H, W] con gli score di calibrazione dello stesso frame/fase.
    alpha:
        Livello di significativita'. Esempio: alpha=0.10.

    Returns
    -------
    threshold_map:
        Array [H, W] con la soglia conformal per ogni pixel.
    q_level:
        Livello di quantile usato.

    Nota
    ----
    Uso la stessa correzione finite-sample gia' usata nel notebook:

        q_level = 1 - alpha + 1 / n_calib

    limitata all'intervallo [0, 1].
    """
    calib_scores_frame = np.asarray(calib_scores_frame, dtype=np.float32)
    if calib_scores_frame.ndim != 3:
        raise ValueError(f"calib_scores_frame deve avere shape [N, H, W], trovata: {calib_scores_frame.shape}")

    n_calib = calib_scores_frame.shape[0]
    if n_calib == 0:
        raise ValueError("Nessuno score di calibrazione disponibile.")

    q_level = min(1.0, max(0.0, 1.0 - float(alpha) + (1.0 / n_calib)))
    threshold_map = np.quantile(calib_scores_frame, q_level, axis=0).astype(np.float32)
    return threshold_map, q_level


def binary_anomaly_mask_from_threshold(score_tensor: np.ndarray, frame_idx: int, alpha: float) -> tuple[np.ndarray, np.ndarray, float]:
    """Restituisce maschera binaria, soglia conformal e q_level per un frame interno."""
    if len(calib_scores) == 0:
        raise ValueError("calib_scores vuoto: eseguire prima split, modello e calibrazione.")

    if frame_idx < 0 or frame_idx >= score_tensor.shape[0]:
        raise IndexError(f"frame_idx fuori range: {frame_idx}, video length = {score_tensor.shape[0]}")

    if frame_idx >= calib_scores.shape[1]:
        raise IndexError(
            f"frame_idx={frame_idx} non presente in calib_scores, che ha {calib_scores.shape[1]} frame/fasi."
        )

    score_frame = score_tensor[frame_idx]
    calib_frame_scores = calib_scores[:, frame_idx, :, :]
    threshold_map, q_level = conformal_quantile_threshold(calib_frame_scores, alpha)
    anomaly_mask = score_frame > threshold_map
    return anomaly_mask, threshold_map, q_level


# -----------------------------
# Widget interattivi
# -----------------------------
slider_alpha_bin = widgets.FloatSlider(
    value=DEFAULT_ALPHA_BINARY,
    min=0.01,
    max=0.50,
    step=0.01,
    description="Alpha conformal:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

drop_prov_bin = widgets.Dropdown(
    options=sorted(list(inference_scores.keys())),
    description="Provino:",
    style={"description_width": "initial"},
)

drop_stroke_bin = widgets.Dropdown(
    options=[],
    description="Passata:",
    style={"description_width": "initial"},
)

slider_frame_bin = widgets.IntSlider(
    min=0,
    max=0,
    step=1,
    value=0,
    description="Frame/fase:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

out_bin = widgets.Output()


def update_stroke_list_bin(*args):
    p = drop_prov_bin.value
    if not p:
        return

    strokes = sorted(list(inference_scores.get(p, {}).keys()))
    old_val = drop_stroke_bin.value
    drop_stroke_bin.options = strokes

    if strokes:
        new_val = old_val if old_val in strokes else strokes[0]
        drop_stroke_bin.value = new_val
        score_tensor = inference_scores[p][new_val]
        slider_frame_bin.max = score_tensor.shape[0] - 1
        slider_frame_bin.value = min(slider_frame_bin.value, slider_frame_bin.max)
        plot_binary_anomalies()
    else:
        slider_frame_bin.max = 0
        with out_bin:
            clear_output(wait=True)
            print(f"Nessuna passata disponibile per il provino {p}.")


def update_frame_range_bin(*args):
    p = drop_prov_bin.value
    s = drop_stroke_bin.value
    if not p or not s:
        return

    score_tensor = inference_scores[p][s]
    slider_frame_bin.max = score_tensor.shape[0] - 1
    slider_frame_bin.value = min(slider_frame_bin.value, slider_frame_bin.max)
    plot_binary_anomalies()


def plot_binary_anomalies(*args):
    p = drop_prov_bin.value
    s = drop_stroke_bin.value
    frame_idx = slider_frame_bin.value
    alpha = float(slider_alpha_bin.value)

    if not p or not s:
        return

    if len(calib_scores) == 0:
        with out_bin:
            clear_output(wait=True)
            print("calib_scores vuoto: eseguire prima split, modello e calibrazione.")
        return

    score_tensor = inference_scores[p][s]
    video_path = Path(OUT_DIR) / str(p) / str(s)
    if not video_path.exists():
        with out_bin:
            clear_output(wait=True)
            print(f"Mini-video non trovato: {video_path}")
        return

    video = np.load(video_path).astype(np.float32)
    frame_data = video[frame_idx]

    anomaly_mask, threshold_map, q_level = binary_anomaly_mask_from_threshold(score_tensor, frame_idx, alpha)

    n_anom = int(np.sum(anomaly_mask))
    n_pixels = int(anomaly_mask.size)
    pct_anom = 100.0 * n_anom / n_pixels if n_pixels else 0.0
    n_calib = int(calib_scores.shape[0])

    # Overlay binario: mascheriamo i pixel non-anomali e coloriamo in rosso pieno quelli anomali.
    # Il valore dell'overlay non rappresenta la gravita': indica solo anomalo / non anomalo.
    from matplotlib.colors import ListedColormap

    overlay_mask = np.ma.masked_where(~anomaly_mask, anomaly_mask.astype(np.float32))
    red_cmap = ListedColormap(["red"])
    red_cmap.set_bad(color=(0, 0, 0, 0))

    with out_bin:
        out_bin.clear_output(wait=True)
        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(13, 5))

            im0 = axes[0].imshow(frame_data, cmap="gray")
            axes[0].set_title("Frame termico originale")
            axes[0].axis("off")
            plt.colorbar(im0, ax=axes[0], label="Temperatura (°C)")

            axes[1].imshow(frame_data, cmap="gray")
            axes[1].imshow(overlay_mask, cmap=red_cmap, alpha=0.90, vmin=0, vmax=1)
            axes[1].set_title("Overlay binario rosso\npixel rosso = anomalo")
            axes[1].axis("off")

            fig.suptitle(
                f"Pixel anomali conformal | Provino {p} | {s} | frame/fase {frame_idx} | "
                f"alpha={alpha:.2f} | {n_anom}/{n_pixels} pixel ({pct_anom:.2f}%)",
                fontsize=12,
            )
            plt.tight_layout()
            display(fig)
            plt.close(fig)

        print("Regola decisionale usata:")
        print(f"  score_test > soglia_conformal")
        print(f"  soglia_conformal = quantile pixel-per-pixel degli score di calibrazione")
        print(f"  alpha = {alpha:.2f}; q_level = {q_level:.4f}; n_calibrazione = {n_calib}")
        print(f"  pixel anomali = {n_anom} su {n_pixels} ({pct_anom:.2f}%)")
        print("Nota: l'overlay rosso e' binario; non rappresenta la distanza dalla soglia.")


drop_prov_bin.observe(update_stroke_list_bin, names="value")
drop_stroke_bin.observe(update_frame_range_bin, names="value")
slider_frame_bin.observe(plot_binary_anomalies, names="value")
slider_alpha_bin.observe(plot_binary_anomalies, names="value")

display(widgets.HBox([drop_prov_bin, drop_stroke_bin, slider_alpha_bin]))
display(slider_frame_bin)
display(out_bin)

update_stroke_list_bin()


IntSlider(value=0, description='Frame/fase:', layout=Layout(width='600px'), max=0, style=SliderStyle(descripti…

Output()

### 11B. Visualizzazione della gravità delle anomalie conformal

Dopo la maschera binaria della cella 11, questa visualizzazione mostra anche **quanto** un pixel supera la soglia conformal.

La logica resta la stessa:

- si calcola lo score di non-conformità del pixel;
- si confronta lo score con la soglia conformal pixel-per-pixel;
- dove il pixel è anomalo, si visualizza la distanza positiva dalla soglia.

Questa cella è quindi utile per leggere la feature di **eccesso oltre soglia conformal**: non conta solo quanti pixel sono anomali, ma evidenzia anche la loro intensità rispetto alla soglia calibrata.

Il valore predefinito è `alpha = 0.10`, ma resta modificabile tramite slider.


In [28]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import os
import numpy as np

slider_pvalue = widgets.FloatSlider(value=0.10, min=0.01, max=0.5, step=0.01, description='P-Value (Alpha):', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))
drop_prov_cp = widgets.Dropdown(options=list(inference_scores.keys()), description='Provino:', style={'description_width': 'initial'})
drop_stroke_cp = widgets.Dropdown(options=[], description='Passata:', style={'description_width': 'initial'})
slider_frame_cp = widgets.IntSlider(min=0, max=0, step=1, value=0, description='Frame interno:', style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))
out_cp = widgets.Output()

def update_stroke_list_cp(*args):
    p = drop_prov_cp.value
    if not p: return
    strokes = list(inference_scores.get(p, {}).keys())
    
    old_val = drop_stroke_cp.value
    drop_stroke_cp.options = strokes
    if strokes:
        slider_frame_cp.max = inference_scores[p][strokes[0]].shape[0] - 1
        new_val = strokes[0]
        drop_stroke_cp.value = new_val
        if old_val == new_val:
            plot_cp()

def plot_cp(*args):
    p = drop_prov_cp.value
    s = drop_stroke_cp.value
    frame = slider_frame_cp.value
    alpha = slider_pvalue.value
    
    if not p or not s: return
    if len(calib_scores) == 0: return
    
    # 1. Calcolo del Quantile q
    n = calib_scores.shape[0]
    q_level = min(1.0, max(0.0, 1.0 - alpha + (1.0 / n)))
    
    # Quantile pixel-by-pixel sul frame corrente
    quantile_threshold = np.quantile(calib_scores[:, frame, :, :], q_level, axis=0) 
    
    # 2. Dati attuali
    video = np.load(os.path.join(OUT_DIR, p, s))
    frame_data = video[frame]
    score_frame = inference_scores[p][s][frame]
    
    # 3. Anomalia
    anomaly_mask = score_frame > quantile_threshold
    
    with out_cp:
        out_cp.clear_output(wait=True)
        with plt.ioff():
            fig, ax = plt.subplots(figsize=(8, 6))
            
            # Base image
            im0 = ax.imshow(frame_data, cmap='gray')
            
            # Overlay mask with magnitude
            anomaly_mag = np.where(anomaly_mask, score_frame - quantile_threshold, np.nan)
            im_overlay = ax.imshow(anomaly_mag, cmap='YlOrRd', alpha=0.75)
            
            ax.set_title(f"Anomalie Sovrapposte (p-value={alpha:.2f}) | {p}/{s} [Frame:{frame}]")
            plt.colorbar(im0, ax=ax, label='Temperatura (\u00b0C)')
            
            # Check if there are anomalies to draw colorbar
            if not np.all(np.isnan(anomaly_mag)):
                plt.colorbar(im_overlay, ax=ax, label='Gravità Anomalia (Distanza dalla Soglia)')
                
            ax.axis('off')
            plt.tight_layout()
            display(fig)
            plt.close(fig)

drop_prov_cp.observe(update_stroke_list_cp, names='value')
drop_stroke_cp.observe(plot_cp, names='value')
slider_frame_cp.observe(plot_cp, names='value')
slider_pvalue.observe(plot_cp, names='value')

display(widgets.HBox([drop_prov_cp, drop_stroke_cp, slider_pvalue]))
display(slider_frame_cp)
display(out_cp)

update_stroke_list_cp()


IntSlider(value=0, description='Frame interno:', layout=Layout(width='600px'), max=0, style=SliderStyle(descri…

Output()

### 13. Visualizzazione Anomalie Mediate (p-value)

Questa visualizzazione mostra le anomalie mediate su tutti i frame di ciascun mini-video. L'anomalia viene calcolata per ogni frame confrontando la non-conformità con la soglia del p-value (Alpha) corrispondente a quel frame, e poi mediata temporalmente. In questo modo viene mostrata la gravità media dell'anomalia lungo l'intera passata, senza la necessità di uno slider per i singoli frame.

In [29]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import os
import numpy as np

slider_pvalue_avg = widgets.FloatSlider(value=0.10, min=0.01, max=0.5, step=0.01, description='P-Value (Alpha):', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))
drop_prov_avg = widgets.Dropdown(options=list(inference_scores.keys()), description='Provino:', style={'description_width': 'initial'})
drop_stroke_avg = widgets.Dropdown(options=[], description='Passata:', style={'description_width': 'initial'})
out_avg = widgets.Output()

def update_stroke_list_avg(*args):
    p = drop_prov_avg.value
    if not p: return
    strokes = list(inference_scores.get(p, {}).keys())
    
    old_val = drop_stroke_avg.value
    drop_stroke_avg.options = strokes
    if strokes:
        new_val = strokes[0]
        drop_stroke_avg.value = new_val
        if old_val == new_val:
            plot_avg()

def plot_avg(*args):
    p = drop_prov_avg.value
    s = drop_stroke_avg.value
    alpha = slider_pvalue_avg.value
    
    if not p or not s: return
    if len(calib_scores) == 0: return
    
    # Carica il mini-video
    video = np.load(os.path.join(OUT_DIR, p, s))
    num_frames = video.shape[0]
    
    # Calcolo del Quantile q
    n = calib_scores.shape[0]
    q_level = min(1.0, max(0.0, 1.0 - alpha + (1.0 / n)))
    
    anomaly_mags = []
    for frame in range(num_frames):
        # Quantile pixel-by-pixel sul frame corrente
        quantile_threshold = np.quantile(calib_scores[:, frame, :, :], q_level, axis=0)
        score_frame = inference_scores[p][s][frame]
        
        # Anomalia per questo frame
        anomaly_mask = score_frame > quantile_threshold
        # Usiamo 0.0 per i pixel non-anomali in modo da fare una media corretta
        anomaly_mag = np.where(anomaly_mask, score_frame - quantile_threshold, 0.0)
        anomaly_mags.append(anomaly_mag)
        
    # Media dell'intensità dell'anomalia su tutti i frame
    mean_anomaly_mag = np.mean(anomaly_mags, axis=0)
    
    # Base image: temperatura media del mini-video
    mean_frame_data = np.mean(video, axis=0)
    
    # Overlay delle anomalie (solo dove la media è strettamente maggiore di 0)
    overlay_data = np.where(mean_anomaly_mag > 0, mean_anomaly_mag, np.nan)
    
    with out_avg:
        out_avg.clear_output(wait=True)
        with plt.ioff():
            fig, ax = plt.subplots(figsize=(8, 6))
            
            # Immagine di base (temperatura media)
            im0 = ax.imshow(mean_frame_data, cmap='gray')
            
            # Overlay con le anomalie
            im_overlay = ax.imshow(overlay_data, cmap='YlOrRd', alpha=0.75)
            
            ax.set_title(f"Anomalie Mediate (p-value={alpha:.2f}) | {p}/{s} (Media su {num_frames} frame)")
            plt.colorbar(im0, ax=ax, label='Temperatura Media (\u00b0C)')
            
            if not np.all(np.isnan(overlay_data)):
                plt.colorbar(im_overlay, ax=ax, label='Gravità Anomalia Media (Distanza dalla Soglia)')
                
            ax.axis('off')
            plt.tight_layout()
            display(fig)
            plt.close(fig)

drop_prov_avg.observe(update_stroke_list_avg, names='value')
drop_stroke_avg.observe(plot_avg, names='value')
slider_pvalue_avg.observe(plot_avg, names='value')

display(widgets.HBox([drop_prov_avg, drop_stroke_avg, slider_pvalue_avg]))
display(out_avg)

update_stroke_list_avg()


Output()

### 14. Evoluzione delle anomalie e delle feature conformal per tutti i provini

Questa cella unifica la visualizzazione originale dell'evoluzione dei pixel anomali con le feature conformal arricchite.

Il menu **Feature** consente di scegliere quale grandezza visualizzare passata per passata:

1. **Conteggio anomalie**: numero totale di elementi anomali nel mini-video della passata. Corrisponde alla lettura originale della sezione 14.
2. **Eccesso oltre soglia conformal**: somma di quanto lo score supera la soglia conformal; misura la gravita' dell'anomalia, non solo la sua presenza.
3. **Deficit termico freddo**: somma delle deviazioni in cui il provino e' piu' freddo del modello normale oltre la soglia conformal.

Lo slider consente di variare il `p-value`/`alpha` conformal. Il valore predefinito e' `0.10`, coerente con l'analisi finale delle prime 20 passate.

**Legenda stile linee:**
- **Colori:** grigio per i provini standard, oro/arancio/rosso/viola per fermi crescenti.
- **Stili linea:** solido, tratteggiato, puntinato e tratto-punto per distinguere le ripetizioni.

Nota: il valore predefinito di alpha è 0.10, ma resta modificabile tramite slider come nella versione precedente.


In [ ]:
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np

# ==========================================================
# Evoluzione per tutti i provini: conteggio + feature conformal
# ==========================================================
# Questa cella sostituisce la vecchia sezione 14 e la sezione 14B.
# Mantiene il grafico di evoluzione per tutti i provini e aggiunge
# la possibilita' di scegliere la feature da visualizzare.
#
# Feature disponibili:
# - count: numero totale di elementi anomali nella passata
# - excess: somma dello score oltre soglia conformal
# - cold: deficit freddo oltre soglia conformal
#
# La feature di persistenza nella passata e' stata rimossa.

try:
    MODEL_INPUT_DIR
except NameError:
    MODEL_INPUT_DIR = Path("dati") / "mini_video_resampled_phase"

OUT_DIR = str(MODEL_INPUT_DIR)

feature_alpha_slider = widgets.FloatSlider(
    value=0.10,
    min=0.01,
    max=0.50,
    step=0.01,
    description="P-Value (Alpha):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

feature_kind_dropdown = widgets.Dropdown(
    options=[
        ("Conteggio anomalie", "count"),
        ("Eccesso oltre soglia", "excess"),
        ("Deficit freddo", "cold"),
    ],
    value="count",
    description="Feature:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

out_feature_evolution = widgets.Output()

prefix_colors = {
    "99": ("gray", "Standard (0s)"),
    "10": ("gold", "Fermo 10s"),
    "30": ("darkorange", "Fermo 30s"),
    "60": ("crimson", "Fermo 60s"),
    "90": ("purple", "Fermo 90s"),
}

style_map = {
    "1": "-",
    "2": "--",
    "3": ":",
    "4": "-.",
}

feature_label_map = {
    "count": "Conteggio anomalie",
    "excess": "Eccesso oltre soglia",
    "cold": "Deficit freddo",
}

ylabel_map = {
    "count": "Numero elementi anomali",
    "excess": "Somma eccesso oltre soglia",
    "cold": "Somma deficit freddo oltre soglia",
}


def compute_feature_for_stroke(score: np.ndarray, threshold: np.ndarray, kind: str, video: np.ndarray | None = None) -> float:
    """Calcola la feature selezionata per una singola passata ricampionata."""
    if score.shape != threshold.shape:
        return np.nan

    anomaly_mask = score > threshold

    if kind == "count":
        return float(np.sum(anomaly_mask))

    if kind == "excess":
        return float(np.sum(np.maximum(score - threshold, 0.0)))

    if kind == "cold":
        if video is None:
            return np.nan
        if mean_model is None or video.shape != mean_model.shape:
            return np.nan
        cold_raw = mean_model - video
        return float(np.sum(np.where(cold_raw > threshold, cold_raw - threshold, 0.0)))

    return np.nan


def plot_feature_evolution(*args):
    alpha = float(feature_alpha_slider.value)
    kind = feature_kind_dropdown.value

    if len(calib_scores) == 0 or mean_model is None:
        with out_feature_evolution:
            clear_output(wait=True)
            print("Modello o calibrazione non disponibili. Eseguire prima le celle 8-10.")
        return

    n = calib_scores.shape[0]
    q_level = min(1.0, max(0.0, 1.0 - alpha + (1.0 / n)))
    threshold = np.quantile(calib_scores, q_level, axis=0)

    with out_feature_evolution:
        clear_output(wait=True)
        with plt.ioff():
            fig, ax = plt.subplots(figsize=(12, 6))

            plotted_any = False
            skipped = []

            for p in sorted(inference_scores.keys()):
                values = []
                p_strokes = sorted(inference_scores[p].keys())

                for s in p_strokes:
                    score = inference_scores[p][s]

                    if score.shape != threshold.shape:
                        values.append(np.nan)
                        skipped.append((p, s, score.shape, threshold.shape, "score shape diversa dalla soglia"))
                        continue

                    video = None
                    if kind == "cold":
                        video_path = Path(OUT_DIR) / str(p) / str(s)
                        if not video_path.exists():
                            values.append(np.nan)
                            skipped.append((p, s, None, threshold.shape, "mini-video non trovato"))
                            continue
                        video = np.load(video_path).astype(np.float32)
                        if video.shape != mean_model.shape:
                            values.append(np.nan)
                            skipped.append((p, s, video.shape, mean_model.shape, "video shape diversa da mean_model"))
                            continue

                    values.append(compute_feature_for_stroke(score, threshold, kind, video=video))

                prefix = str(p)[:2]
                rep = str(p)[2] if len(str(p)) > 2 else "1"
                color, desc = prefix_colors.get(prefix, ("blue", "Unknown"))
                linestyle = style_map.get(rep, "-")

                if len(values) > 0:
                    plotted_any = True
                    ax.plot(
                        range(1, len(values) + 1),
                        values,
                        color=color,
                        linestyle=linestyle,
                        linewidth=2,
                        label=f"Provino {p} ({desc}, Rep {rep})",
                    )

            ax.set_title(
                f"Evoluzione {feature_label_map[kind]} per passata "
                f"(alpha={alpha:.2f}, q={q_level:.4f})"
            )
            ax.set_xlabel("Passate (strokes)")
            ax.set_ylabel(ylabel_map[kind])
            ax.grid(True, linestyle="--", alpha=0.5)

            if plotted_any:
                ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.0)

            plt.tight_layout()
            display(fig)
            plt.close(fig)

        if skipped:
            print(f"Nota: {len(skipped)} passate saltate per incompatibilita' o file mancante.")
            print("Prime 5 passate saltate:")
            for item in skipped[:5]:
                print("  ", item)


feature_alpha_slider.observe(plot_feature_evolution, names="value")
feature_kind_dropdown.observe(plot_feature_evolution, names="value")

display(widgets.HBox([feature_alpha_slider, feature_kind_dropdown]))
display(out_feature_evolution)

plot_feature_evolution()



Output()

### 15. Export Dati Finali (Conformal ricampionato)

Salva su disco i risultati finali dell'analisi conformal basata sul dataset ricampionato in fase normalizzata.

Per non sovrascrivere i risultati della pipeline originale, questa copia salva in:

`dati/dataset_finale/conformal_resampled_phase/`

Output principali:

- `mean_model.npy`: modello medio sul dataset ricampionato;
- `calib_scores.npy`: score di calibrazione;
- `anomaly_counts.csv`: anomalie per singola passata;
- `anomaly_summary.csv`: statistiche aggregate per provino;
- `resampling_metadata.csv`: tracciabilità del ricampionamento di ogni stroke;
- `resampling_config.json`: parametri usati per il ricampionamento.


In [31]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

try:
    EXPORT_DIR_RESAMPLED
except NameError:
    EXPORT_DIR_RESAMPLED = Path("dati") / "dataset_finale" / "conformal_resampled_phase"

try:
    MODEL_INPUT_DIR
except NameError:
    MODEL_INPUT_DIR = Path("dati") / "mini_video_resampled_phase"

EXPORT_DIR = Path(EXPORT_DIR_RESAMPLED)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# Export arricchito: oltre al conteggio binario, salviamo 3 nuove famiglie di feature
# =============================================================================
# 1) excess: quanto lo score supera la soglia conformal.
# 2) cold_deficit: quota direzionale fredda, cioe' quanto T osservata e' piu' bassa
#    del modello normale, oltre la soglia conformal.
# 3) persistence/AUC: quanto l'anomalia persiste dentro la passata, misurata come
#    frazione di fasi temporali significativamente anomale e come area temporale.
#
# Nota: il conteggio originale rimane invariato per compatibilita' con the_end.ipynb.

# 1. Salva il modello medio e gli score di calibrazione
if mean_model is not None:
    np.save(EXPORT_DIR / "mean_model.npy", mean_model)
    print(f"Salvato mean_model.npy ({mean_model.shape})")
else:
    raise RuntimeError("mean_model vuoto: eseguire prima training e calibrazione.")

if len(calib_scores) > 0:
    np.save(EXPORT_DIR / "calib_scores.npy", calib_scores)
    print(f"Salvato calib_scores.npy ({calib_scores.shape})")
else:
    raise RuntimeError("calib_scores vuoto: eseguire prima split e calibrazione.")

# 2. Copia metadata/config del ricampionamento nell'export finale, se disponibili
resampling_meta_path = Path(MODEL_INPUT_DIR) / "resampling_metadata.csv"
resampling_config_path = Path(MODEL_INPUT_DIR) / "resampling_config.json"
if resampling_meta_path.exists():
    resampling_meta = pd.read_csv(resampling_meta_path)
    resampling_meta.to_csv(EXPORT_DIR / "resampling_metadata.csv", index=False)
    print(f"Salvato resampling_metadata.csv ({len(resampling_meta)} righe)")
else:
    resampling_meta = None
    print("[INFO] resampling_metadata.csv non trovato.")

if resampling_config_path.exists():
    with open(resampling_config_path, "r", encoding="utf-8") as f:
        resampling_config = json.load(f)
    with open(EXPORT_DIR / "resampling_config.json", "w", encoding="utf-8") as f:
        json.dump(resampling_config, f, indent=4)

# 3. Calcolo metriche conformal a diversi p-value
P_VALUES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
ACTIVE_PHASE_FRACTION_THRESHOLD = 0.05  # una fase e' persistente se almeno il 5% dei pixel e' anomalo
rows = []
n_calib = calib_scores.shape[0]
n_elements = int(np.prod(mean_model.shape))
n_pixels_per_phase = int(np.prod(mean_model.shape[1:])) if mean_model.ndim == 3 else np.nan

# Pre-calcolo dei quantili per ogni p-value
quantile_cache = {}
for alpha in P_VALUES:
    q_level = min(1.0, max(0.0, 1.0 - alpha + (1.0 / n_calib)))
    quantile_cache[alpha] = np.quantile(calib_scores, q_level, axis=0)

# Helper per recuperare il mini-video reale, necessario per l'anomalia direzionale fredda.
def load_video_for_score(provino_id, stroke_file):
    video_path = Path(MODEL_INPUT_DIR) / str(provino_id) / str(stroke_file)
    if not video_path.exists():
        return None
    return np.load(video_path).astype(np.float32)

for p in tqdm(sorted(inference_scores.keys()), desc="Export anomalie arricchite"):
    p_strokes = sorted(list(inference_scores[p].keys()))
    for stroke_idx, s in enumerate(p_strokes):
        score = inference_scores[p][s]
        video = load_video_for_score(p, s)

        row = {
            "provino_id": p,
            "stroke_file": s,
            "stroke_idx": stroke_idx,
            "model_input_dir": str(MODEL_INPUT_DIR),
            "n_elements": n_elements,
            "n_pixels_per_phase": n_pixels_per_phase,
        }

        # Aggiungo info sulla lunghezza originale se disponibile
        if resampling_meta is not None:
            meta_match = resampling_meta[(resampling_meta["provino_id"].astype(str) == str(p)) & (resampling_meta["stroke_file"] == s)]
            if len(meta_match) == 1:
                row["original_length"] = int(meta_match.iloc[0]["original_length"])
                row["target_length"] = int(meta_match.iloc[0]["target_length"])
                row["compression_factor"] = float(meta_match.iloc[0]["compression_factor"])

        for alpha in P_VALUES:
            threshold = quantile_cache[alpha]
            suffix = f"pv{alpha}"

            if score.shape != threshold.shape:
                row[f"anomaly_{suffix}"] = 0
                row[f"anomaly_fraction_{suffix}"] = 0.0
                row[f"excess_sum_{suffix}"] = 0.0
                row[f"excess_mean_all_{suffix}"] = 0.0
                row[f"excess_mean_anom_{suffix}"] = 0.0
                row[f"cold_deficit_sum_{suffix}"] = 0.0
                row[f"cold_deficit_mean_all_{suffix}"] = 0.0
                row[f"cold_deficit_mean_anom_{suffix}"] = 0.0
                row[f"active_phase_count_{suffix}"] = 0
                row[f"active_phase_fraction_{suffix}"] = 0.0
                row[f"auc_anomaly_fraction_{suffix}"] = 0.0
                row[f"max_phase_anomaly_fraction_{suffix}"] = 0.0
                continue

            anomaly_mask = score > threshold
            n_anomaly = int(np.sum(anomaly_mask))
            anomaly_fraction = float(n_anomaly / n_elements) if n_elements else 0.0

            # 1) Gravita': somma dell'eccesso oltre la soglia conformal.
            excess = np.maximum(score - threshold, 0.0).astype(np.float32)
            excess_sum = float(np.sum(excess))
            excess_mean_all = float(excess_sum / n_elements) if n_elements else 0.0
            excess_mean_anom = float(excess_sum / n_anomaly) if n_anomaly else 0.0

            # 2) Direzione fredda: T_model - T_video, solo se supera la stessa soglia conformal.
            #    Se il video non e' caricabile, lasciamo NaN per non inventare informazione.
            if video is not None and video.shape == mean_model.shape:
                cold_raw = mean_model - video
                cold_excess = np.where(cold_raw > threshold, cold_raw - threshold, 0.0).astype(np.float32)
                cold_deficit_sum = float(np.sum(cold_excess))
                cold_count = int(np.sum(cold_excess > 0))
                cold_deficit_mean_all = float(cold_deficit_sum / n_elements) if n_elements else 0.0
                cold_deficit_mean_anom = float(cold_deficit_sum / cold_count) if cold_count else 0.0
            else:
                cold_deficit_sum = np.nan
                cold_deficit_mean_all = np.nan
                cold_deficit_mean_anom = np.nan

            # 3) Persistenza dentro la passata: quante fasi temporali hanno almeno una quota
            #    significativa di pixel anomali + area sotto la curva della frazione anomala.
            phase_counts = np.sum(anomaly_mask, axis=(1, 2)).astype(float)
            phase_fraction = phase_counts / float(n_pixels_per_phase) if n_pixels_per_phase else phase_counts * np.nan
            active_phase_mask = phase_fraction >= ACTIVE_PHASE_FRACTION_THRESHOLD
            active_phase_count = int(np.sum(active_phase_mask))
            active_phase_fraction = float(active_phase_count / len(phase_fraction)) if len(phase_fraction) else 0.0
            auc_anomaly_fraction = float(np.trapezoid(phase_fraction, dx=1.0)) if len(phase_fraction) > 1 else float(np.sum(phase_fraction))
            max_phase_anomaly_fraction = float(np.max(phase_fraction)) if len(phase_fraction) else 0.0

            row[f"anomaly_{suffix}"] = n_anomaly
            row[f"anomaly_fraction_{suffix}"] = anomaly_fraction
            row[f"excess_sum_{suffix}"] = excess_sum
            row[f"excess_mean_all_{suffix}"] = excess_mean_all
            row[f"excess_mean_anom_{suffix}"] = excess_mean_anom
            row[f"cold_deficit_sum_{suffix}"] = cold_deficit_sum
            row[f"cold_deficit_mean_all_{suffix}"] = cold_deficit_mean_all
            row[f"cold_deficit_mean_anom_{suffix}"] = cold_deficit_mean_anom
            row[f"active_phase_count_{suffix}"] = active_phase_count
            row[f"active_phase_fraction_{suffix}"] = active_phase_fraction
            row[f"auc_anomaly_fraction_{suffix}"] = auc_anomaly_fraction
            row[f"max_phase_anomaly_fraction_{suffix}"] = max_phase_anomaly_fraction

        rows.append(row)

df_counts = pd.DataFrame(rows)
df_counts.to_csv(EXPORT_DIR / "anomaly_counts.csv", index=False)
print(f"Salvato anomaly_counts.csv arricchito ({len(df_counts)} righe)")

# 4. Aggregati per provino: mantiene i vecchi aggregati e aggiunge le nuove famiglie di feature.
metric_prefixes = [
    "anomaly",
    "anomaly_fraction",
    "excess_sum",
    "excess_mean_all",
    "excess_mean_anom",
    "cold_deficit_sum",
    "cold_deficit_mean_all",
    "cold_deficit_mean_anom",
    "active_phase_count",
    "active_phase_fraction",
    "auc_anomaly_fraction",
    "max_phase_anomaly_fraction",
]

agg_rows = []
for p in sorted(inference_scores.keys()):
    sub = df_counts[df_counts["provino_id"] == p]
    row = {"provino_id": p, "n_passate": len(sub)}
    for alpha in P_VALUES:
        suffix = f"pv{alpha}"
        for metric in metric_prefixes:
            col = f"{metric}_{suffix}"
            if col not in sub.columns:
                continue
            row[f"mean_{col}"] = sub[col].mean()
            row[f"max_{col}"] = sub[col].max()
            row[f"total_{col}"] = sub[col].sum()
            row[f"std_{col}"] = sub[col].std()
    agg_rows.append(row)

df_summary = pd.DataFrame(agg_rows)
df_summary.to_csv(EXPORT_DIR / "anomaly_summary.csv", index=False)
print(f"Salvato anomaly_summary.csv arricchito ({len(df_summary)} righe)")

# 5. Config di export arricchito
feature_config = {
    "description": "Export conformal arricchito con conteggio, gravita' oltre soglia, deficit freddo e persistenza temporale.",
    "p_values": P_VALUES,
    "active_phase_fraction_threshold": ACTIVE_PHASE_FRACTION_THRESHOLD,
    "model_input_dir": str(MODEL_INPUT_DIR),
    "export_dir": str(EXPORT_DIR),
    "feature_meanings": {
        "anomaly": "conteggio binario originale degli elementi anomali",
        "anomaly_fraction": "conteggio normalizzato su L*H*W",
        "excess_sum": "somma di max(score - soglia_conformal, 0)",
        "cold_deficit_sum": "somma della sola anomalia fredda: max((mean_model - video) - soglia_conformal, 0)",
        "active_phase_fraction": "frazione di fasi con almeno il 5% dei pixel anomali",
        "auc_anomaly_fraction": "area sotto la curva della frazione di pixel anomali dentro la passata"
    }
}
with open(EXPORT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=4)

print("\nExport completato in:", EXPORT_DIR)
print("Nuove colonne principali per alpha=0.10:")
for col in [
    "anomaly_pv0.1",
    "anomaly_fraction_pv0.1",
    "excess_sum_pv0.1",
    "cold_deficit_sum_pv0.1",
    "active_phase_fraction_pv0.1",
    "auc_anomaly_fraction_pv0.1",
]:
    if col in df_counts.columns:
        print(" -", col)


Salvato mean_model.npy ((16, 30, 68))
Salvato calib_scores.npy ((47, 16, 30, 68))
Salvato resampling_metadata.csv (1073 righe)


Export anomalie arricchite:   0%|          | 0/11 [00:00<?, ?it/s]

Salvato anomaly_counts.csv arricchito (1073 righe)
Salvato anomaly_summary.csv arricchito (11 righe)

Export completato in: dati\dataset_finale\conformal_resampled_phase
Nuove colonne principali per alpha=0.10:
 - anomaly_pv0.1
 - anomaly_fraction_pv0.1
 - excess_sum_pv0.1
 - cold_deficit_sum_pv0.1
 - active_phase_fraction_pv0.1
 - auc_anomaly_fraction_pv0.1
